# Kaggriculture Competition Agent

## Overview

This notebook contains a complete agent for the Kaggle Kaggriculture competition, a two player turn based farming simulation where each player manages a 10 by 10 farm grid over 720 turns (30 days times 24 turns per day) and competes to maximize bank coins at season end. The agent replays a precomputed 720 step action trace that encodes an optimized farming strategy, then applies four reactive overlays that adapt to the live game state without disrupting the underlying schedule.

## The Precomputed Trace

The trace encodes a full season of actions for the farmer and all hired hands, plus market orders at every step. It opens by purchasing 3 cows and 6 melon seeds on turn 0, builds pastures and places animals over the first several days, ramps labor from 4 hands on day 0 to 11 hands by day 10, buys two land expansions on days 8 and 11, and rotates from melons into a strawberry engine alongside a mixed herd of cows and sheep. Over the full season the trace issues approximately 294 hire orders, 44 seed purchases, 7 animal purchases, 2 land purchases, 370 wheat buy orders for animal feed, and 350 sell orders diversified across wheat, fertilizer, strawberries, milk, wool, and melons.

## Overlay 1: Hand Count Adaptation

The trace assumes a specific number of hands are hired each day. If a market buy order consumes a slot that was intended for a hire, fewer hands may be available than the trace expects. This overlay pads the hands action list with PASS actions when more hands exist than expected, or trims the list when fewer are available. Without this overlay, the simulator would silently drop actions for non existent hands or raise errors for missing hand actions.

## Overlay 2: Terminal Liquidation

Starting at step 696 (day 29, hour 0), this overlay fills any remaining slots in the 10 order market list with sell orders for shed inventory. Items are prioritized by descending price times quantity so the most valuable sellable goods are listed first. Premium goods (strawberry, melon, milk, wool) are sold in batches of at most 5 units to avoid crashing the price to the floor, while staple goods are sold in batches of at most 20. This ensures no sellable inventory remains in the shed at game end, since unsold items contribute zero to the final score.

## Overlay 3: Terminal Harvesting

Also starting at step 696, this overlay scans the farmer and hand actions for the terminal phase and replaces any PASS action with a more productive alternative. If a unit is carrying inventory while standing on a shed access tile, the action becomes DROP. If the unit is standing on a tile with positive yield units, the action becomes HARVEST. If harvestable tiles remain and the unit is empty, it moves toward the nearest one. If the unit is carrying inventory but not at the shed, it moves toward the nearest shed access tile. This captures yield that the trace did not plan to harvest, increasing total sellable inventory in the final liquidation.

## Overlay 4: Opponent Exposure Modeling

This helper function counts the opponent animal and crop tiles to estimate their production capacity per item. Each cow contributes 6 units of milk exposure, each sheep contributes 6 units of wool exposure, each goose contributes 4 units of egg exposure, and each crop contributes 4 to 6 units of its respective product exposure. This information feeds into the terminal liquidation priority ordering, ensuring items with high opponent production are sold first before the opponent can crash the price.

## Design Philosophy

The agent preserves the trace wherever possible. All four overlays are additive: they only modify actions when the trace action is a no-op (PASS), only add market orders when there is room in the 10 slot list, and never remove or reorder existing tape orders. This conservative approach avoids disrupting the carefully tuned economic flow of the trace while still adapting to the live game state in the final hours. The agent scores approximately 172,000 coins against a passive opponent and wins the majority of head to head games against all reference agents across multiple seeds.

## Usage

Run the Python cell below to write the agent to main.py, then submit main.py to the Kaggle competition. The file is self-contained with no external dependencies beyond the Python standard library.

In [ ]:
%%writefile main.py
"""Kaggriculture competition agent.

Architecture: trace replay with adaptive overlays.
  1. Hand count adaptation (pad/trim to actual hand count)
  2. Sell reordering by priority (price times quantity plus opponent pressure)
  3. Terminal liquidation (fill remaining market slots with shed items)
  4. Terminal harvesting (redirect idle hands to harvest nearby crops)

The agent replays a precomputed 720 step action sequence that encodes an
optimized farming strategy: 10 melon tiles and 15 wheat tiles in the NW
quadrant, a mixed herd of cows and sheep, two land expansions, and a
ramp from 4 to 11 hired hands. Four reactive overlays layer on top of
the trace to adapt to the live game state without disrupting the
underlying schedule.

Overlay 1 (hand count adaptation) ensures the hands action list always
matches the actual number of hired hands. If the trace planned for 8
hands but only 6 were hired (because a buy crowded out a hire), the
list is trimmed to 6 and the extra actions are dropped. Conversely, if
more hands were hired than the trace expected, the list is padded with
PASS actions so every hand gets an instruction.

Overlay 2 (sell reordering) sorts the SELL orders within the 10 slot
market list by descending priority, where priority equals price times
quantity plus opponent exposure times sensitivity times price. This
surfaces the highest value sells first so they are processed before
any per unit price erosion from concurrent opponent sells. Non SELL
orders (BUY_PRODUCT, HIRE, BUY_SEED, etc.) stay in their original
positions and are never displaced.

Overlay 3 (terminal liquidation) activates at step 696 (day 29 hour 0).
It fills any remaining slots in the market list with SELL orders for
shed inventory, sorted by descending price times quantity. This
ensures no sellable inventory is left in the shed at game end, since
unsold items contribute zero to the final score.

Overlay 4 (terminal harvesting) also activates at step 696. It scans
the farmer and hand actions for the terminal phase and replaces any
PASS action with a more productive alternative: HARVEST if standing on
a tile with yield units, DROP if carrying inventory while standing on a
shed access tile, or MOVE toward the nearest harvestable tile. This
captures yield that the trace did not plan to harvest, increasing the
total sellable inventory in the final liquidation.
"""

import copy

# === TRACE ACTIONS (720 precomputed steps) ===
TRACE_ACTIONS = [{'farmer': ['PASS'], 'hands': [], 'market': [['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 3], ['BUY_SEED', 'MELON', 6]]}, {'farmer': ['BUILD_PASTURE'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['BUY_ANIMAL', 'SHEEP', 1]]}, {'farmer': ['PICKUP', 'COW', 1], 'hands': [['PICKUP', 'SHEEP', 1], ['NORTH'], ['WEST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2], ['HIRE']]}, {'farmer': ['PLACE', 'COW'], 'hands': [['WEST'], ['BUILD_PASTURE'], ['NORTH'], ['WEST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['BUILD_PASTURE'], ['NORTH'], ['SOUTH'], ['PICKUP', 'COW', 1]], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['NORTH'], ['BUILD_PASTURE'], ['PICKUP', 'COW', 1], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['NORTH'], ['NORTH'], ['NORTH'], ['PASS']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['NORTH'], ['PLACE', 'COW'], ['PASS']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['NORTH'], 'hands': [['PLACE', 'SHEEP'], ['PLANT', 'MELON'], ['CARE'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['WATER'], ['NORTH'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['PICKUP', 'WHEAT', 2]], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['NORTH'], ['PLANT', 'MELON'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['NORTH'], ['WATER'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['WEST'], ['WEST'], ['SOUTH'], ['FEED']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['PLANT', 'MELON'], ['PLANT', 'MELON'], ['NORTH'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['WATER'], ['NORTH'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['NORTH'], ['PLANT', 'MELON'], ['SOUTH'], ['PLACE', 'COW']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WATER'], ['SOUTH'], ['FEED']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['EAST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['PLANT', 'MELON'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['WATER'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['SOUTH'], ['PASS'], ['PASS']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['CARE'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['DROP'], ['SOUTH'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['SOUTH'], ['DROP'], ['PASS'], ['SOUTH'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['BUY_SEED', 'MELON', 1]]}, {'farmer': ['FEED'], 'hands': [['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['PLANT', 'MELON'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WHEAT', 4], ['SELL', 'FERTILIZER', 1], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['CARE'], ['SOUTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['EAST'], ['NORTH'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['NORTH'], ['WEST'], ['DROP'], ['NORTH'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['WATER'], ['PASS'], ['WATER'], ['PASS']], 'market': [['BUY_SEED', 'MELON', 1]]}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['WEST'], ['PASS'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['PLANT', 'MELON'], ['PASS'], ['WATER'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['WATER'], ['PASS'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['DROP'], ['NORTH'], ['PASS'], ['WEST'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['WATER'], ['PASS'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['WEST'], 'hands': [['PASS'], ['SOUTH'], ['PASS'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['FEED'], 'hands': [['PASS'], ['SOUTH'], ['PASS'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['EAST'], 'hands': [['PASS'], ['SOUTH'], ['PASS'], ['SOUTH'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['SOUTH'], ['PASS'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['EAST'], ['PASS'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['EAST'], ['PASS'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['DROP'], ['PASS'], ['EAST'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS']], 'market': [['BUY_SEED', 'STRAWBERRY', 2]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WHEAT', 4], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['NORTH'], ['NORTH'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['NORTH'], ['DROP'], ['WEST'], ['NORTH'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['PASS'], ['WEST'], ['WATER'], ['PASS']], 'market': [['BUY_SEED', 'MELON', 2]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['PASS'], ['WEST'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['PASS'], ['SOUTH'], ['WATER'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['PLANT', 'STRAWBERRY'], ['PASS'], ['PLANT', 'MELON'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['PASS'], ['WATER'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['PASS'], ['SOUTH'], ['NORTH'], ['PASS']], 'market': []}, {'farmer': ['EAST'], 'hands': [['PLANT', 'MELON'], ['PASS'], ['SOUTH'], ['WATER'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['EAST'], ['EAST'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['EAST'], ['WATER'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['EAST'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['DROP'], ['SOUTH'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['PASS'], ['EAST'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 4], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['NORTH'], ['NORTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['NORTH'], ['DROP'], ['NORTH'], ['NORTH'], ['PASS'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['PASS'], ['WEST'], ['WATER'], ['PASS'], ['PASS']], 'market': [['BUY_SEED', 'STRAWBERRY', 2]]}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['PASS'], ['WEST'], ['WEST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['PASS'], ['WEST'], ['WATER'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['PLANT', 'STRAWBERRY'], ['NORTH'], ['WATER'], ['WEST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['SOUTH'], ['WEST'], ['SOUTH'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['WATER'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['WEST'], 'hands': [['DIG'], ['PASS'], ['SOUTH'], ['NORTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['FEED'], 'hands': [['PLANT', 'STRAWBERRY'], ['PASS'], ['WATER'], ['WATER'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WATER'], ['PASS'], ['EAST'], ['SOUTH'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['EAST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['SOUTH'], ['WATER'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WEST'], ['PASS'], ['SOUTH'], ['SOUTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['SOUTH'], ['SOUTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['EAST'], ['SOUTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['EAST'], ['EAST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'FERTILIZER', 2], ['SELL', 'WHEAT', 4], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['WEST'], ['CARE'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['NORTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['NORTH'], ['DROP'], ['PLANT', 'STRAWBERRY'], ['NORTH'], ['PASS'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['PASS'], ['WATER'], ['WATER'], ['PASS'], ['PASS']], 'market': [['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['PASS'], ['WEST'], ['WEST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['WEST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['PASS'], ['NORTH'], ['WATER'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['WEST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WATER'], ['PASS'], ['WEST'], ['WATER'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['WEST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WATER'], ['PASS'], ['SOUTH'], ['WATER'], ['PASS'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['SOUTH'], ['SOUTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['EAST'], ['SOUTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['PASS'], ['EAST'], ['SOUTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['EAST'], ['SOUTH'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['EAST'], ['EAST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['DROP'], ['EAST'], ['PASS'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WHEAT', 4], ['SELL', 'FERTILIZER', 1], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['SOUTH'], ['NORTH'], ['DROP'], ['WATER'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['WEST'], ['PASS'], ['SOUTH'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WEST'], ['WATER'], ['PASS'], ['SOUTH'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['WEST'], ['PASS'], ['EAST'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH'], ['PASS'], ['DROP'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'WOOL', 6]]}, {'farmer': ['SOUTH'], 'hands': [['PLANT', 'STRAWBERRY'], ['WEST'], ['PASS'], ['PASS'], ['WEST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_ANIMAL', 'COW', 2], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['WATER'], ['NORTH'], ['WEST'], ['NORTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WEST'], ['WEST'], ['WEST'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WATER'], ['WATER'], ['BUILD_PASTURE'], ['BUILD_PASTURE'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['SOUTH'], ['WEST'], ['EAST'], ['WATER'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'COW', 2], 'hands': [['WATER'], ['SOUTH'], ['WATER'], ['EAST'], ['SOUTH'], ['SOUTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['EAST'], ['WEST'], ['PASS'], ['SOUTH'], ['SOUTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['EAST'], ['SOUTH'], ['SOUTH'], ['PASS'], ['SOUTH'], ['EAST']], 'market': []}, {'farmer': ['PLACE', 'COW'], 'hands': [['WATER'], ['WATER'], ['EAST'], ['PASS'], ['EAST'], ['EAST']], 'market': []}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['EAST'], ['EAST'], ['PASS'], ['EAST'], ['EAST']], 'market': []}, {'farmer': ['CARE'], 'hands': [['SOUTH'], ['SOUTH'], ['EAST'], ['PASS'], ['EAST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'FERTILIZER', 3], ['SELL', 'WHEAT', 6], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 6], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'COW', 1], ['CARE'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 4], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['CARE'], ['EAST'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['SOUTH'], ['NORTH'], ['NORTH'], ['DROP'], ['WATER'], ['WEST']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['SOUTH'], ['NORTH'], ['NORTH'], ['PASS'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['WEST'], ['WATER'], ['WATER'], ['PASS'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['WEST'], ['WEST'], ['PASS'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['FEED'], 'hands': [['PLACE', 'COW'], ['WATER'], ['NORTH'], ['PASS'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['CARE'], ['WEST'], ['WATER'], ['PASS'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['WEST'], ['SOUTH'], ['PICKUP', 'WHEAT', 1], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['WATER'], ['WEST'], ['WEST'], ['SOUTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['DROP'], ['WEST'], ['WATER'], ['FEED'], ['EAST'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['FEED'], 'hands': [['PASS'], ['WATER'], ['WEST'], ['EAST'], ['PASS'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['PASS'], ['WATER'], ['WATER'], ['PASS'], ['PASS'], ['SOUTH']], 'market': []}, {'farmer': ['FEED'], 'hands': [['PASS'], ['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['PASS'], ['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['PASS'], ['WATER'], ['EAST'], ['PASS'], ['PASS'], ['EAST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['SOUTH'], ['EAST'], ['PASS'], ['PASS'], ['EAST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['EAST'], ['EAST'], ['PASS'], ['PASS'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WHEAT', 6], ['SELL', 'FERTILIZER', 3], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 6], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['HARVEST'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 4], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['DROP'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']], 'market': [['SELL', 'MILK', 6], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['HARVEST'], ['WEST'], ['NORTH']], 'market': [['BUY_ANIMAL', 'SHEEP', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['HARVEST'], ['CARE'], ['SOUTH'], ['CARE'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 4], ['HIRE']]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['EAST'], ['NORTH'], ['DROP'], ['COLLECT_FERTILIZER'], ['WEST'], ['WEST']], 'market': [['SELL', 'MILK', 6], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['FEED'], ['DROP'], ['CARE'], ['WEST'], ['WEST'], ['CARE'], ['WEST']], 'market': [['SELL', 'MILK', 6]]}, {'farmer': ['NORTH'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['WEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2], ['BUY_LAND']]}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['EAST'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_ANIMAL', 'SHEEP', 2], ['BUY_SEED', 'STRAWBERRY', 5]]}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['EAST'], ['SOUTH'], ['EAST'], ['NORTH'], ['NORTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['WEST'], 'hands': [['BUILD_PASTURE'], ['BUILD_PASTURE'], ['EAST'], ['EAST'], ['NORTH'], ['WATER'], ['NORTH']], 'market': [['SELL', 'WHEAT', 3]]}, {'farmer': ['FEED'], 'hands': [['NORTH'], ['EAST'], ['BUILD_PASTURE'], ['PICKUP', 'SHEEP', 2], ['WATER'], ['NORTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['BUILD_PASTURE'], ['NORTH'], ['PICKUP', 'SHEEP', 1], ['NORTH'], ['WATER'], ['WEST']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['WEST'], ['SOUTH'], ['PICKUP', 'SHEEP', 1], ['WATER'], ['EAST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PASS'], 'hands': [['PLANT', 'STRAWBERRY'], ['PASS'], ['NORTH'], ['EAST'], ['WEST'], ['WATER'], ['WATER']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['NORTH'], ['PLACE', 'SHEEP'], ['WATER'], ['SOUTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['CARE'], ['SOUTH'], ['NORTH'], ['WEST'], ['SOUTH'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['PLANT', 'STRAWBERRY'], ['PASS'], ['NORTH'], ['PLACE', 'SHEEP'], ['WATER'], ['WEST'], ['WEST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['PASS'], ['EAST'], ['CARE'], ['SOUTH'], ['WATER'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['PASS'], ['SOUTH'], ['NORTH'], ['WATER'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['SOUTH'], 'hands': [['PLANT', 'STRAWBERRY'], ['PASS'], ['SOUTH'], ['PLACE', 'SHEEP'], ['SOUTH'], ['SOUTH'], ['EAST']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['SOUTH'], ['CARE'], ['EAST'], ['EAST'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['PASS'], ['SOUTH'], ['SOUTH'], ['WATER'], ['DROP'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 2], ['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WHEAT', 7], ['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 7], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 7], ['BUY_PRODUCT', 'WHEAT', 2], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'SHEEP', 1], ['CARE'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 7], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['EAST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['EAST'], ['CARE'], ['WEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['PLACE', 'SHEEP'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WEST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['SOUTH'], ['PICKUP', 'WHEAT', 1], ['CARE'], ['CARE'], ['CARE'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['NORTH'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['FEED'], 'hands': [['DROP'], ['NORTH'], ['PASS'], ['WEST'], ['NORTH'], ['WEST'], ['WEST'], ['EAST'], ['NORTH'], ['SOUTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['PASS'], ['FEED'], ['PASS'], ['WEST'], ['WATER'], ['CARE'], ['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['WEST'], ['SOUTH'], ['PASS'], ['WATER'], ['WEST'], ['NORTH'], ['NORTH'], ['EAST'], ['WEST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['PICKUP', 'WHEAT', 2], ['WEST'], ['PASS'], ['NORTH'], ['WATER'], ['NORTH'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['FEED'], ['PASS'], ['WATER'], ['WEST'], ['WEST'], ['NORTH'], ['WATER'], ['WEST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['FEED'], ['SOUTH'], ['PASS'], ['NORTH'], ['WATER'], ['WATER'], ['WATER'], ['EAST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['EAST'], ['PASS'], ['NORTH'], ['SOUTH'], ['NORTH'], ['EAST'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['FEED'], ['EAST'], ['PASS'], ['WATER'], ['WEST'], ['WEST'], ['WATER'], ['WATER'], ['SOUTH'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['PASS'], ['PASS'], ['SOUTH'], ['WATER'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['PASS'], ['PASS'], ['WATER'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['WATER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['EAST'], ['WATER'], ['SOUTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['WATER'], ['EAST'], ['DROP'], ['SOUTH'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['SOUTH'], ['EAST'], ['PASS'], ['SOUTH'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['EAST'], ['EAST'], ['PASS'], ['SOUTH'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['EAST'], ['EAST'], ['PASS'], ['WEST'], ['DROP'], ['SOUTH']], 'market': [['SELL', 'WOOL', 4], ['SELL', 'FERTILIZER', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WHEAT', 10], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['CARE'], ['EAST'], ['HARVEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 8], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['CARE'], ['HARVEST'], ['NORTH'], ['HARVEST'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['FEED'], ['COLLECT_FERTILIZER'], ['WEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['WEST'], ['CARE'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['NORTH'], ['FEED'], ['FEED'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['NORTH'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WATER'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['EAST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['FEED'], ['WEST'], ['WATER'], ['NORTH'], ['NORTH'], ['NORTH'], ['EAST'], ['PICKUP', 'WHEAT', 2], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['PLANT', 'STRAWBERRY'], ['SOUTH'], ['NORTH'], ['WEST'], ['SOUTH'], ['WATER'], ['WATER'], ['EAST'], ['EAST'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['WATER'], ['WEST'], ['WATER'], ['WEST'], ['EAST'], ['NORTH'], ['HARVEST'], ['WATER'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['FEED'], ['NORTH'], ['WATER'], ['NORTH'], ['WEST'], ['SOUTH'], ['WEST'], ['FEED'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['EAST'], ['WATER'], ['WEST'], ['WEST'], ['NORTH'], ['SOUTH'], ['WEST'], ['WEST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['SOUTH'], 'hands': [['PLANT', 'STRAWBERRY'], ['EAST'], ['HARVEST'], ['WATER'], ['WEST'], ['NORTH'], ['SOUTH'], ['WEST'], ['DROP'], ['EAST']], 'market': [['SELL', 'MILK', 3], ['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['PASS'], ['SOUTH'], ['NORTH'], ['WEST'], ['WATER'], ['SOUTH'], ['WEST'], ['PASS'], ['DROP']], 'market': [['SELL', 'MELON', 6]]}, {'farmer': ['PASS'], 'hands': [['NORTH'], ['PASS'], ['SOUTH'], ['NORTH'], ['WATER'], ['HARVEST'], ['DROP'], ['WEST'], ['PASS'], ['PASS']], 'market': [['SELL', 'MELON', 6], ['SELL', 'MILK', 3], ['BUY_ANIMAL', 'COW', 3], ['BUY_SEED', 'MELON', 5]]}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['NORTH'], ['SOUTH'], ['WATER'], ['HARVEST'], ['SOUTH'], ['PASS'], ['PLANT', 'MELON'], ['EAST'], ['PASS']], 'market': [['BUY_PRODUCT', 'WHEAT', 6], ['BUY_ANIMAL', 'SHEEP', 2]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['EAST'], ['SOUTH'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['WATER'], ['EAST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4], ['BUY_SEED', 'MELON', 1]]}, {'farmer': ['BUILD_PASTURE'], 'hands': [['WEST'], ['EAST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WEST'], ['WEST'], ['BUILD_PASTURE'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['SOUTH'], ['EAST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WEST'], ['PLANT', 'MELON'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['BUILD_PASTURE'], 'hands': [['BUILD_PASTURE'], ['WEST'], ['DROP'], ['SOUTH'], ['SOUTH'], ['EAST'], ['BUILD_PASTURE'], ['WATER'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'MELON', 6], ['BUY_SEED', 'MELON', 1]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['PASS'], ['PICKUP', 'COW', 3], ['SOUTH'], ['EAST'], ['EAST'], ['EAST'], ['WEST'], ['PASS'], ['PICKUP', 'SHEEP', 2]], 'market': [['SELL', 'FERTILIZER', 8]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'MELON', 18], ['SELL', 'WHEAT', 15], ['SELL', 'MILK', 3], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'COW', 3], ['NORTH'], ['PICKUP', 'SHEEP', 2], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['CARE'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 4]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['NORTH'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['NORTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['FEED'], ['NORTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WEST'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['WEST'], ['NORTH'], ['CARE'], ['NORTH'], ['EAST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['CARE'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['FEED'], ['FEED'], ['EAST'], ['NORTH'], ['PLACE', 'SHEEP'], ['NORTH'], ['WATER'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['NORTH'], ['SOUTH'], ['EAST'], ['EAST'], ['CARE'], ['WATER'], ['EAST'], ['COLLECT_FERTILIZER'], ['WEST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['FEED'], ['SOUTH'], ['PLACE', 'COW'], ['WATER'], ['SOUTH'], ['HARVEST'], ['EAST'], ['NORTH'], ['WEST'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['EAST'], ['PICKUP', 'WHEAT', 2], ['CARE'], ['SOUTH'], ['NORTH'], ['SOUTH'], ['EAST'], ['NORTH'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['EAST'], ['EAST'], ['NORTH'], ['EAST'], ['EAST'], ['SOUTH'], ['WATER'], ['NORTH'], ['WEST'], ['PLANT', 'MELON']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['WEST'], ['FEED'], ['FEED'], ['SOUTH'], ['PLANT', 'MELON'], ['PLACE', 'SHEEP'], ['SOUTH'], ['EAST'], ['NORTH'], ['WATER'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['WEST'], 'hands': [['WATER'], ['NORTH'], ['EAST'], ['SOUTH'], ['WATER'], ['CARE'], ['DROP'], ['WATER'], ['WEST'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'MELON', 6], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['FEED'], ['FEED'], ['SOUTH'], ['NORTH'], ['EAST'], ['NORTH'], ['SOUTH'], ['WEST'], ['WATER'], ['WATER']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WATER'], ['EAST'], ['WEST'], ['EAST'], ['WEST'], ['EAST'], ['SOUTH'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['EAST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['FEED'], ['PASS'], ['PLACE', 'COW'], ['WEST'], ['PLANT', 'MELON'], ['PASS'], ['SOUTH'], ['WATER'], ['WATER'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['CARE'], ['WEST'], ['WATER'], ['PASS'], ['PLANT', 'MELON'], ['EAST'], ['SOUTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1], ['BUY_LAND']]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['FEED'], ['PASS'], ['WEST'], ['PLANT', 'MELON'], ['SOUTH'], ['PASS'], ['WATER'], ['EAST'], ['EAST'], ['WEST']], 'market': [['BUY_SEED', 'STRAWBERRY', 4]]}, {'farmer': ['SOUTH'], 'hands': [['DIG'], ['EAST'], ['PASS'], ['WEST'], ['WATER'], ['NORTH'], ['PASS'], ['WEST'], ['EAST'], ['WATER'], ['WEST']], 'market': [['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['PASS'], 'hands': [['PLANT', 'STRAWBERRY'], ['FEED'], ['PASS'], ['DROP'], ['SOUTH'], ['WEST'], ['PASS'], ['WEST'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'FERTILIZER', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['PICKUP', 'COW', 1], 'hands': [['WATER'], ['SOUTH'], ['PASS'], ['PASS'], ['WEST'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['EAST'], ['DROP']], 'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['PASS'], 'hands': [['EAST'], ['FEED'], ['PASS'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['PASS'], ['WATER'], ['SOUTH'], ['EAST'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 6], ['BUY_SEED', 'STRAWBERRY', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WHEAT', 14], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'COW', 1], ['CARE'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['WEST'], ['NORTH'], ['EAST'], ['CARE'], ['NORTH'], ['HARVEST'], ['WEST'], ['NORTH'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['FEED'], ['FEED'], ['COLLECT_FERTILIZER'], ['FEED'], ['NORTH'], ['CARE'], ['NORTH'], ['CARE'], ['EAST'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['HARVEST'], ['NORTH'], ['NORTH'], ['HARVEST'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['CARE'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WEST'], ['FEED'], ['EAST'], ['COLLECT_FERTILIZER'], ['FEED'], ['WEST'], ['WATER'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['CARE']], 'market': []}, {'farmer': ['FEED'], 'hands': [['EAST'], ['FEED'], ['SOUTH'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['EAST'], ['WATER'], ['NORTH'], ['NORTH'], ['CARE'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WATER'], ['CARE'], ['SOUTH'], ['EAST'], ['WEST'], ['EAST'], ['WEST'], ['EAST'], ['WEST'], ['SOUTH'], ['CARE'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['EAST'], ['DIG'], ['CARE'], ['WEST'], ['EAST'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['PLANT', 'STRAWBERRY']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['NORTH'], ['WEST'], ['FEED'], ['PLANT', 'STRAWBERRY'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['WATER'], ['WEST'], ['NORTH'], ['WATER']], 'market': [['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['EAST'], 'hands': [['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['WATER'], ['NORTH'], ['WEST'], ['SOUTH'], ['WEST'], ['WEST'], ['WEST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['EAST'], ['PLACE', 'COW'], ['FEED'], ['WEST'], ['SOUTH'], ['WATER'], ['WATER'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['PLANT', 'STRAWBERRY']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WATER'], ['SOUTH'], ['CARE'], ['EAST'], ['WEST'], ['NORTH'], ['SOUTH'], ['EAST'], ['WEST'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2], ['BUY_SEED', 'STRAWBERRY', 2]]}, {'farmer': ['FEED'], 'hands': [['HARVEST'], ['SOUTH'], ['SOUTH'], ['FEED'], ['FERTILIZE'], ['WATER'], ['WATER'], ['WATER'], ['WATER'], ['WEST'], ['WATER'], ['NORTH']], 'market': [['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['EAST'], ['WATER'], ['EAST'], ['EAST'], ['NORTH'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['WEST'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['WEST'], ['WATER'], ['SOUTH'], ['WATER'], ['WATER'], ['WATER'], ['WEST'], ['NORTH']], 'market': [['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['FEED'], ['WATER'], ['SOUTH'], ['FERTILIZE'], ['EAST'], ['SOUTH'], ['EAST'], ['SOUTH'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['NORTH']], 'market': [['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['NORTH'], ['EAST'], ['WEST'], ['SOUTH'], ['WATER'], ['EAST'], ['WATER'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['WEST'], 'hands': [['EAST'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WATER'], ['SOUTH'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 3], ['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['WEST'], 'hands': [['DROP'], ['EAST'], ['WATER'], ['WEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['EAST'], ['SOUTH'], ['WATER'], ['SOUTH']], 'market': [['SELL', 'MELON', 6], ['SELL', 'FERTILIZER', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['DROP'], 'hands': [['PASS'], ['WATER'], ['NORTH'], ['DROP'], ['WATER'], ['WEST'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['EAST'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['SOUTH']], 'market': [['SELL', 'MILK', 3], ['SELL', 'WHEAT', 7], ['SELL', 'FERTILIZER', 8]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'MILK', 6], ['SELL', 'WOOL', 4], ['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 7], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 2]]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 8], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 6], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 8], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['CARE'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['EAST'], ['NORTH'], ['FEED'], ['CARE'], ['EAST'], ['CARE'], ['NORTH'], ['EAST'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['FEED'], ['FEED'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['FEED'], ['FEED'], ['WEST'], ['FEED'], ['EAST'], ['NORTH'], ['CARE'], ['WEST'], ['EAST'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['FEED'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['FERTILIZE'], ['WEST'], ['NORTH'], ['CARE'], ['SOUTH'], ['NORTH'], ['CARE'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['CARE'], ['SOUTH'], ['WEST'], ['WATER'], ['WEST'], ['WEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['COLLECT_FERTILIZER'], ['WEST'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['FERTILIZE'], ['NORTH'], ['NORTH'], ['WEST'], ['WATER'], ['SOUTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['SOUTH'], ['WEST'], ['WEST'], ['EAST'], ['WATER'], ['WEST'], ['WATER'], ['WEST'], ['CARE'], ['SOUTH'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['WATER'], ['FEED'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH'], ['WATER'], ['SOUTH'], ['WATER'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['NORTH'], ['SOUTH'], ['FEED'], ['WEST'], ['HARVEST'], ['WATER'], ['SOUTH'], ['NORTH'], ['WEST'], ['WEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['SOUTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WATER'], ['WATER'], ['SOUTH'], ['FEED'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WEST'], ['HARVEST'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 2], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['SOUTH'], ['EAST'], ['SOUTH'], ['DROP'], ['WATER'], ['WATER'], ['SOUTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['WATER'], ['SOUTH'], ['WATER'], ['WEST'], ['SOUTH'], ['EAST'], ['PASS'], ['PASS'], ['HARVEST'], ['NORTH'], ['WEST'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['SOUTH'], 'hands': [['NORTH'], ['SOUTH'], ['SOUTH'], ['DROP'], ['SOUTH'], ['EAST'], ['PASS'], ['PASS'], ['SOUTH'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['WATER'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['SOUTH'], ['DROP'], ['PASS'], ['PASS'], ['SOUTH'], ['SOUTH'], ['WATER'], ['WATER']], 'market': [['SELL', 'MELON', 6], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['SOUTH'], ['WATER'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['SOUTH']], 'market': [['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['WEST'], ['WEST'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['EAST'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['FEED'], 'hands': [['WATER'], ['DIG'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['DIG'], ['PASS'], ['PASS'], ['PASS'], ['EAST'], ['SOUTH'], ['WATER'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 10], ['BUY_SEED', 'STRAWBERRY', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'MELON', 6], ['SELL', 'STRAWBERRY', 4], ['SELL', 'WHEAT', 14], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'MELON', 1]]}, {'farmer': ['FEED'], 'hands': [['HARVEST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['CARE'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['EAST'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['CARE'], ['EAST'], ['HARVEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['NORTH'], ['COLLECT_FERTILIZER'], ['WEST'], ['EAST'], ['EAST'], ['WEST'], ['HARVEST'], ['CARE'], ['NORTH'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['FEED'], ['EAST'], ['FEED'], ['CARE'], ['FEED'], ['WATER'], ['WEST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['CARE'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['EAST'], ['NORTH'], ['WEST'], ['NORTH'], ['WEST'], ['WEST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['FEED'], ['NORTH'], ['WEST'], ['WATER'], ['FEED'], ['WATER'], ['NORTH'], ['HARVEST'], ['CARE'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['CARE'], ['CARE'], ['FEED'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WATER'], ['CARE'], ['WATER'], ['WATER'], ['NORTH'], ['WEST'], ['SOUTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['EAST'], ['WEST'], ['SOUTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['SOUTH'], ['FERTILIZE'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['FEED'], ['CARE'], ['WATER'], ['EAST'], ['EAST'], ['WATER'], ['WATER'], ['SOUTH'], ['WEST'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['WEST'], ['WATER'], ['CARE'], ['SOUTH'], ['NORTH'], ['EAST'], ['FERTILIZE'], ['SOUTH'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['FEED'], ['EAST'], ['NORTH'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WATER'], ['DROP'], ['NORTH'], ['DROP'], ['CARE']], 'market': [['SELL', 'MILK', 9], ['SELL', 'WOOL', 5]]}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['FEED'], ['WEST'], ['WEST'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['SOUTH'], ['WEST'], ['WATER'], ['WEST'], ['WATER'], ['NORTH'], ['WATER'], ['WEST'], ['HARVEST'], ['SOUTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['FEED'], ['WEST'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['SOUTH'], ['WEST'], ['WEST'], ['SOUTH'], ['SOUTH'], ['CARE']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['DROP'], ['HARVEST'], ['EAST'], ['PASS'], ['WATER'], ['SOUTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['SOUTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['EAST'], ['PASS'], ['WEST'], ['SOUTH'], ['SOUTH'], ['EAST'], ['WATER'], ['SOUTH'], ['SOUTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WEST'], ['PASS'], ['SOUTH'], ['EAST'], ['PASS'], ['WATER'], ['SOUTH'], ['EAST'], ['EAST'], ['WEST'], ['PLANT', 'STRAWBERRY'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 1]]}, {'farmer': ['PASS'], 'hands': [['DROP'], ['PASS'], ['SOUTH'], ['EAST'], ['PASS'], ['WEST'], ['PLANT', 'STRAWBERRY'], ['WATER'], ['DROP'], ['NORTH'], ['WATER'], ['PLANT', 'STRAWBERRY']], 'market': [['SELL', 'WOOL', 6], ['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['EAST'], ['DROP'], ['PASS'], ['WATER'], ['WATER'], ['SOUTH'], ['PASS'], ['WEST'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 5], ['BUY_SEED', 'WHEAT', 2]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['DROP'], ['PASS'], ['PASS'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['PASS'], ['WATER'], ['PLANT', 'WHEAT'], ['SOUTH']], 'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 5]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WOOL', 6], ['SELL', 'WHEAT', 15], ['SELL', 'STRAWBERRY', 2], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['EAST'], ['PICKUP', 'WHEAT', 2], ['SOUTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['CARE'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['WEST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['FEED'], ['WEST'], ['NORTH'], ['CARE'], ['SOUTH'], ['EAST'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['CARE'], ['FEED'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['CARE'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['CARE'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['EAST'], ['WEST'], ['NORTH'], ['EAST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['FEED'], ['FEED'], ['FEED'], ['WATER'], ['FEED'], ['WATER'], ['WEST'], ['HARVEST'], ['WEST'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['HARVEST'], ['CARE'], ['NORTH'], ['CARE'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['PLANT', 'WHEAT'], ['CARE'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['NORTH'], ['WEST'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['EAST'], ['WEST'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['EAST'], ['FEED'], ['WATER'], ['EAST'], ['NORTH'], ['NORTH'], ['SOUTH'], ['EAST'], ['WATER'], ['NORTH'], ['WATER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['FEED'], ['NORTH'], ['SOUTH'], ['WATER'], ['SOUTH'], ['WEST'], ['SOUTH'], ['CARE'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['SOUTH'], ['NORTH'], ['NORTH'], ['WEST'], ['EAST'], ['WEST'], ['WATER'], ['EAST'], ['SOUTH'], ['PLANT', 'STRAWBERRY'], ['SOUTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['WEST'], ['FEED'], ['WATER'], ['HARVEST'], ['WATER'], ['NORTH'], ['SOUTH'], ['NORTH'], ['EAST'], ['WATER'], ['SOUTH'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['EAST'], ['WEST'], ['WEST'], ['SOUTH'], ['WATER'], ['WEST'], ['SOUTH'], ['CARE'], ['EAST'], ['EAST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['HARVEST'], ['FEED'], ['WATER'], ['HARVEST'], ['WATER'], ['SOUTH'], ['DIG'], ['NORTH'], ['COLLECT_FERTILIZER'], ['PLANT', 'STRAWBERRY'], ['CARE'], ['SOUTH']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WEST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['WEST'], ['SOUTH'], ['PLANT', 'WHEAT'], ['SOUTH'], ['EAST'], ['WATER'], ['COLLECT_FERTILIZER'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['HARVEST'], ['FEED'], ['SOUTH'], ['FERTILIZE'], ['WATER'], ['CARE'], ['WATER'], ['SOUTH'], ['EAST'], ['EAST'], ['EAST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['EAST'], ['EAST'], ['WATER'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['WEST'], ['WATER'], ['PLANT', 'WHEAT'], ['WATER'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['FEED'], ['EAST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['PLANT', 'MELON'], ['PASS'], ['WEST'], ['WATER'], ['EAST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['WATER'], ['EAST'], ['EAST'], ['EAST'], ['WEST'], ['CARE'], ['WATER'], ['PASS'], ['SOUTH'], ['EAST'], ['WATER'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PASS'], 'hands': [['NORTH'], ['WATER'], ['EAST'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['NORTH'], ['PASS'], ['WEST'], ['DIG'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['SOUTH'], ['DROP'], ['EAST'], ['WEST'], ['WATER'], ['NORTH'], ['PASS'], ['WEST'], ['NORTH'], ['WEST'], ['WEST']], 'market': [['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 10], ['BUY_SEED', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WOOL', 10], ['SELL', 'FERTILIZER', 4], ['SELL', 'STRAWBERRY', 8], ['SELL', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['HARVEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['EAST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['FEED'], ['FEED'], ['CARE'], ['NORTH'], ['HARVEST'], ['WATER'], ['CARE'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['CARE'], ['NORTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['HARVEST'], 'hands': [['FEED'], ['COLLECT_FERTILIZER'], ['FEED'], ['FEED'], ['EAST'], ['CARE'], ['WATER'], ['NORTH'], ['PLANT', 'WHEAT'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WEST'], ['NORTH'], ['CARE'], ['CARE'], ['EAST'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['WEST'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['FEED'], 'hands': [['NORTH'], ['CARE'], ['NORTH'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['FEED'], ['WATER'], ['EAST'], ['NORTH'], ['WEST'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WATER'], ['WEST'], ['EAST'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['SOUTH'], ['WATER'], ['NORTH'], ['WEST'], ['WEST'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['WEST'], ['FEED'], ['WATER'], ['EAST'], ['SOUTH'], ['FEED'], ['SOUTH'], ['SOUTH'], ['NORTH'], ['WATER'], ['WATER'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['CARE'], ['SOUTH'], ['EAST'], ['EAST'], ['NORTH'], ['WATER'], ['SOUTH'], ['WEST'], ['NORTH'], ['SOUTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['WATER'], ['COLLECT_FERTILIZER'], ['WEST'], ['EAST'], ['CARE'], ['WEST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['HARVEST'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': [['SELL', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['WEST'], ['SOUTH'], ['WEST'], ['FEED'], ['WEST'], ['WEST'], ['WATER'], ['WEST'], ['WEST'], ['NORTH'], ['SOUTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['WATER'], ['SOUTH'], ['WEST'], ['SOUTH'], ['WEST'], ['WEST'], ['EAST'], ['WEST'], ['WATER'], ['WEST'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['SOUTH'], ['HARVEST'], ['WEST'], ['WEST'], ['WEST'], ['WATER'], ['WEST'], ['WEST'], ['WATER'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['FEED'], 'hands': [['FERTILIZE'], ['WEST'], ['WATER'], ['DROP'], ['SOUTH'], ['SOUTH'], ['NORTH'], ['WEST'], ['WATER'], ['WEST'], ['WATER'], ['WATER']], 'market': [['SELL', 'MILK', 6], ['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['WATER'], ['WEST'], ['PASS'], ['SOUTH'], ['WEST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['FERTILIZE'], ['WEST'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['FEED'], 'hands': [['WATER'], ['SOUTH'], ['WATER'], ['PASS'], ['SOUTH'], ['WATER'], ['EAST'], ['EAST'], ['WATER'], ['NORTH'], ['HARVEST'], ['WATER']], 'market': [['SELL', 'WHEAT', 4], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['FERTILIZE'], ['SOUTH'], ['PASS'], ['SOUTH'], ['WEST'], ['PASS'], ['EAST'], ['WEST'], ['SOUTH'], ['WATER'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['WATER'], 'hands': [['WATER'], ['WATER'], ['SOUTH'], ['PASS'], ['SOUTH'], ['WATER'], ['PASS'], ['PASS'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['NORTH'], ['EAST'], ['PASS'], ['FERTILIZE'], ['SOUTH'], ['PASS'], ['PASS'], ['WATER'], ['SOUTH'], ['EAST'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 1], ['SELL', 'FERTILIZER', 8], ['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 3], ['SELL', 'STRAWBERRY', 6], ['SELL', 'WHEAT', 14], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['HARVEST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['PICKUP', 'WHEAT', 2], ['SOUTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['HARVEST'], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['PICKUP', 'FERTILIZER', 3], ['SOUTH'], ['NORTH'], ['CARE'], ['NORTH'], ['NORTH'], ['EAST']], 'market': [['SELL', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH'], ['NORTH'], ['WEST'], ['EAST'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['EAST'], ['NORTH'], ['FEED'], ['HARVEST'], ['FEED'], ['WATER'], ['CARE'], ['CARE'], ['CARE'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['FEED'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['EAST'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['WATER'], ['WEST'], ['WATER'], ['WEST'], ['CARE'], ['CARE'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['FEED'], ['EAST'], ['FEED'], ['NORTH'], ['FEED'], ['SOUTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['EAST'], ['EAST'], ['NORTH'], ['WEST'], ['CARE'], ['WATER'], ['NORTH'], ['WEST'], ['WEST'], ['HARVEST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['FEED'], ['WATER'], ['FERTILIZE'], ['WATER'], ['HARVEST'], ['WEST'], ['WEST'], ['WEST'], ['HARVEST'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['HARVEST'], ['NORTH'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['FERTILIZE'], ['WEST'], ['SOUTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['FEED'], ['EAST'], ['WATER'], ['NORTH'], ['WEST'], ['WEST'], ['WEST'], ['SOUTH'], ['WATER'], ['HARVEST'], ['SOUTH'], ['NORTH']], 'market': []}, {'farmer': ['FEED'], 'hands': [['CARE'], ['FEED'], ['FERTILIZE'], ['NORTH'], ['WATER'], ['CARE'], ['WATER'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['COLLECT_FERTILIZER'], ['SOUTH'], ['EAST'], ['EAST'], ['WEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['WEST'], ['WATER'], ['SOUTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['NORTH'], ['WEST'], ['WATER'], ['EAST'], ['WATER'], ['SOUTH'], ['EAST'], ['HARVEST'], ['WEST'], ['EAST'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['WEST'], ['EAST'], ['FERTILIZE'], ['WEST'], ['SOUTH'], ['WATER'], ['WATER'], ['HARVEST'], ['EAST'], ['WEST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['FERTILIZE'], ['WEST'], ['WATER'], ['WATER'], ['WATER'], ['SOUTH'], ['WEST'], ['WEST'], ['SOUTH'], ['EAST'], ['DROP'], ['NORTH']], 'market': [['SELL', 'WOOL', 6]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WATER'], ['WEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WEST'], ['SOUTH'], ['EAST'], ['EAST'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['WEST'], ['WATER'], ['SOUTH'], ['WATER'], ['EAST'], ['WATER'], ['WATER'], ['WATER'], ['SOUTH'], ['EAST'], ['WATER']], 'market': []}, {'farmer': ['CARE'], 'hands': [['WATER'], ['WEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['FERTILIZE'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['DROP'], ['COLLECT_FERTILIZER'], ['SOUTH']], 'market': [['SELL', 'STRAWBERRY', 4]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['FEED'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['NORTH'], ['WATER'], ['EAST'], ['EAST'], ['PASS'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['EAST'], ['SOUTH'], ['DROP'], ['SOUTH'], ['WATER'], ['NORTH'], ['WATER'], ['WATER'], ['PASS'], ['WEST'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 9], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WOOL', 18], ['SELL', 'MILK', 3], ['SELL', 'STRAWBERRY', 4], ['SELL', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['WEST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['HARVEST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['SOUTH'], ['WEST'], ['EAST'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['SOUTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['EAST'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['PICKUP', 'WHEAT', 2], ['FEED'], ['FEED'], ['CARE'], ['FEED'], ['WATER'], ['SOUTH'], ['WATER'], ['NORTH'], ['EAST'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['NORTH'], ['HARVEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['WATER'], ['WEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['NORTH'], ['WEST'], ['HARVEST'], ['NORTH'], ['FEED'], ['WATER'], ['SOUTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['FEED'], ['FEED'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['NORTH'], ['NORTH'], ['WATER'], ['WEST'], ['EAST'], ['CARE'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['CARE'], ['CARE'], ['WEST'], ['COLLECT_FERTILIZER'], ['CARE'], ['WATER'], ['NORTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['EAST'], ['HARVEST'], ['FEED'], ['NORTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['SOUTH'], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['FEED'], ['SOUTH'], ['CARE'], ['WEST'], ['NORTH'], ['CARE'], ['WEST'], ['EAST'], ['NORTH'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['EAST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['NORTH'], ['EAST'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['NORTH'], ['FEED'], ['EAST'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['SOUTH'], ['HARVEST'], ['NORTH'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['SOUTH'], ['DROP'], ['FERTILIZE'], ['WATER'], ['WATER'], ['FERTILIZE'], ['SOUTH'], ['WATER'], ['EAST'], ['FERTILIZE'], ['HARVEST']], 'market': [['SELL', 'MILK', 6], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['FERTILIZE'], ['FEED'], ['NORTH'], ['WATER'], ['SOUTH'], ['WEST'], ['WATER'], ['NORTH'], ['SOUTH'], ['WATER'], ['SOUTH'], ['EAST']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WATER'], ['EAST'], ['NORTH'], ['NORTH'], ['HARVEST'], ['WEST'], ['WEST'], ['WEST'], ['WEST'], ['EAST'], ['SOUTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['FEED'], ['WEST'], ['EAST'], ['SOUTH'], ['WATER'], ['HARVEST'], ['WATER'], ['WATER'], ['FERTILIZE'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WATER'], ['EAST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['SOUTH'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WATER'], ['WATER'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['SOUTH'], ['NORTH'], ['WEST'], ['EAST'], ['WATER'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['SOUTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['WATER'], ['EAST'], ['WATER'], ['WATER'], ['EAST'], ['SOUTH'], ['WATER'], ['EAST'], ['NORTH'], ['SOUTH'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['WEST'], ['DROP'], ['WATER'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['WATER'], ['EAST']], 'market': [['SELL', 'WOOL', 4], ['SELL', 'MILK', 3], ['SELL', 'STRAWBERRY', 2]]}, {'farmer': ['PASS'], 'hands': [['WATER'], ['WEST'], ['SOUTH'], ['WATER'], ['PASS'], ['SOUTH'], ['WATER'], ['NORTH'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'FERTILIZER', 8]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WOOL', 4], ['SELL', 'STRAWBERRY', 10], ['SELL', 'MILK', 3], ['SELL', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['SOUTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['WEST'], ['FEED'], ['SOUTH'], ['NORTH'], ['CARE'], ['NORTH'], ['EAST'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['CARE'], ['FEED'], ['COLLECT_FERTILIZER'], ['WEST'], ['WEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['NORTH'], ['NORTH'], ['FEED'], ['WATER'], ['NORTH'], ['NORTH'], ['CARE'], ['NORTH'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['FEED'], ['CARE'], ['FEED'], ['CARE'], ['HARVEST'], ['SOUTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['CARE'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FERTILIZE'], ['WEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['HARVEST'], ['EAST'], ['EAST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['WEST'], ['WATER'], ['NORTH'], ['CARE'], ['WEST'], ['WATER'], ['EAST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WATER'], ['WEST'], ['FEED'], ['WEST'], ['WEST'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['HARVEST'], ['WEST'], ['CARE'], ['CARE']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['FEED'], ['NORTH'], ['WATER'], ['WEST'], ['NORTH'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['NORTH'], ['HARVEST'], ['NORTH'], ['WEST'], ['WEST'], ['WATER'], ['EAST'], ['SOUTH'], ['HARVEST'], ['FERTILIZE'], ['SOUTH']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WATER'], ['EAST'], ['WEST'], ['WEST'], ['WATER'], ['WATER'], ['NORTH'], ['EAST'], ['SOUTH'], ['NORTH'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['EAST'], ['FERTILIZE'], ['HARVEST'], ['NORTH'], ['SOUTH'], ['WEST'], ['HARVEST'], ['WEST'], ['NORTH'], ['WATER'], ['WEST']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['EAST'], ['EAST'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['HARVEST'], ['EAST'], ['DROP'], ['NORTH'], ['EAST'], ['DROP']], 'market': [['SELL', 'MILK', 12]]}, {'farmer': ['SOUTH'], 'hands': [['FERTILIZE'], ['FEED'], ['EAST'], ['WATER'], ['NORTH'], ['SOUTH'], ['EAST'], ['HARVEST'], ['NORTH'], ['WATER'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WATER'], ['SOUTH'], ['SOUTH'], ['WEST'], ['SOUTH'], ['SOUTH'], ['EAST'], ['SOUTH'], ['SOUTH'], ['EAST'], ['HARVEST'], ['SOUTH']], 'market': []}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['EAST'], ['EAST'], ['WATER'], ['WATER'], ['SOUTH'], ['HARVEST'], ['SOUTH'], ['PASS'], ['EAST'], ['WATER'], ['SOUTH']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['FEED'], ['EAST'], ['SOUTH'], ['EAST'], ['NORTH'], ['SOUTH'], ['WATER'], ['PASS'], ['WATER'], ['SOUTH'], ['NORTH']], 'market': [['SELL', 'FERTILIZER', 3]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['SOUTH'], ['DROP'], ['EAST'], ['EAST'], ['WATER'], ['EAST'], ['SOUTH'], ['PASS'], ['WEST'], ['SOUTH'], ['PASS']], 'market': [['SELL', 'STRAWBERRY', 2]]}, {'farmer': ['WEST'], 'hands': [['WATER'], ['FEED'], ['PASS'], ['WATER'], ['WATER'], ['NORTH'], ['EAST'], ['WATER'], ['PASS'], ['WATER'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['SOUTH'], ['WEST'], ['PASS'], ['SOUTH'], ['SOUTH'], ['WATER'], ['WATER'], ['SOUTH'], ['PASS'], ['SOUTH'], ['WATER'], ['PASS']], 'market': [['SELL', 'FERTILIZER', 6]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 12], ['SELL', 'MILK', 3], ['SELL', 'WHEAT', 18], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['HARVEST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['EAST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['CARE'], ['PICKUP', 'FERTILIZER', 3], ['HARVEST'], ['CARE'], ['WEST'], ['EAST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['HARVEST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WEST'], ['EAST'], ['EAST'], ['NORTH'], ['FEED'], ['WATER'], ['CARE'], ['WATER'], ['CARE'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['FEED'], ['COLLECT_FERTILIZER'], ['FEED'], ['HARVEST'], ['HARVEST'], ['WEST'], ['EAST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['CARE'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['CARE'], ['EAST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WATER'], ['WATER'], ['WEST'], ['NORTH'], ['WEST'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['HARVEST'], ['EAST'], ['FEED'], ['NORTH'], ['FEED'], ['NORTH'], ['NORTH'], ['WATER'], ['CARE'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['HARVEST'], ['NORTH'], ['WATER'], ['WEST'], ['WEST'], ['WEST'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['WATER'], ['NORTH'], ['HARVEST'], ['WATER'], ['HARVEST'], ['NORTH'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 1], 'hands': [['HARVEST'], ['FEED'], ['WATER'], ['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['HARVEST'], ['CARE'], ['NORTH'], ['WEST'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['COLLECT_FERTILIZER'], ['CARE'], ['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['HARVEST'], ['NORTH'], ['FERTILIZE']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['FERTILIZE'], ['HARVEST'], ['EAST'], ['WEST'], ['EAST'], ['SOUTH'], ['WEST'], ['NORTH'], ['FERTILIZE'], ['WATER']], 'market': []}, {'farmer': ['FEED'], 'hands': [['EAST'], ['WEST'], ['WATER'], ['NORTH'], ['WATER'], ['DIG'], ['WATER'], ['WEST'], ['WATER'], ['WEST'], ['WATER'], ['NORTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['EAST'], ['FEED'], ['NORTH'], ['HARVEST'], ['EAST'], ['NORTH'], ['SOUTH'], ['WEST'], ['SOUTH'], ['NORTH'], ['EAST'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['FEED'], ['HARVEST'], ['WATER'], ['SOUTH'], ['WATER'], ['WATER'], ['WATER'], ['NORTH'], ['WATER'], ['WEST'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['EAST'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['HARVEST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['FERTILIZE'], ['EAST'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['EAST'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WATER'], ['WEST'], ['WATER'], ['WATER'], ['HARVEST'], ['SOUTH']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['EAST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WEST'], ['SOUTH'], ['EAST'], ['WATER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['FEED'], ['DROP'], ['WATER'], ['WEST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['WEST'], ['WEST'], ['EAST'], ['EAST'], ['WATER']], 'market': [['SELL', 'MILK', 12], ['SELL', 'FERTILIZER', 4]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['PASS'], ['SOUTH'], ['DROP'], ['WEST'], ['SOUTH'], ['CARE'], ['WEST'], ['WATER'], ['WATER'], ['WATER'], ['SOUTH']], 'market': [['SELL', 'STRAWBERRY', 4], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['PASS'], ['WEST'], ['EAST'], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 7]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'WOOL', 20], ['SELL', 'MELON', 12], ['SELL', 'MILK', 6], ['SELL', 'STRAWBERRY', 12], ['SELL', 'WHEAT', 14], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 1], ['CARE'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['EAST'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['HARVEST'], ['CARE'], ['SOUTH'], ['NORTH'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['FEED'], ['FEED'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['NORTH'], ['CARE'], ['NORTH'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['NORTH']], 'market': []}, {'farmer': ['WEST'], 'hands': [['WEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH'], ['FEED'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['CARE'], ['WEST'], ['NORTH'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['FEED'], ['WEST'], ['FEED'], ['COLLECT_FERTILIZER'], ['CARE'], ['EAST'], ['NORTH'], ['EAST'], ['NORTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['COLLECT_FERTILIZER'], ['CARE'], ['EAST'], ['EAST'], ['WEST'], ['FEED'], ['FERTILIZE'], ['CARE'], ['WEST'], ['SOUTH'], ['NORTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['HARVEST'], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['CARE'], ['EAST'], ['HARVEST'], ['EAST'], ['WATER'], ['HARVEST'], ['WEST'], ['WEST'], ['EAST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['DIG'], ['EAST'], ['COLLECT_FERTILIZER'], ['FEED'], ['NORTH'], ['CARE'], ['EAST'], ['COLLECT_FERTILIZER'], ['WATER'], ['WATER'], ['WATER'], ['EAST']], 'market': []}, {'farmer': ['FEED'], 'hands': [['WEST'], ['EAST'], ['EAST'], ['HARVEST'], ['DIG'], ['COLLECT_FERTILIZER'], ['NORTH'], ['EAST'], ['SOUTH'], ['WEST'], ['EAST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['CARE'], 'hands': [['WEST'], ['EAST'], ['CARE'], ['SOUTH'], ['NORTH'], ['NORTH'], ['WEST'], ['FERTILIZE'], ['SOUTH'], ['WATER'], ['WATER'], ['HARVEST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['WEST'], ['EAST'], ['COLLECT_FERTILIZER'], ['FEED'], ['FERTILIZE'], ['EAST'], ['WATER'], ['WATER'], ['WEST'], ['NORTH'], ['HARVEST'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['FEED'], ['EAST'], ['NORTH'], ['NORTH'], ['EAST'], ['HARVEST'], ['EAST'], ['HARVEST'], ['NORTH'], ['SOUTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FERTILIZE'], 'hands': [['SOUTH'], ['EAST'], ['HARVEST'], ['NORTH'], ['EAST'], ['WATER'], ['SOUTH'], ['FERTILIZE'], ['NORTH'], ['WEST'], ['SOUTH'], ['EAST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WATER'], 'hands': [['WATER'], ['FEED'], ['EAST'], ['NORTH'], ['HARVEST'], ['NORTH'], ['SOUTH'], ['WATER'], ['SOUTH'], ['WATER'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['NORTH'], ['WATER'], ['FERTILIZE'], ['WEST'], ['HARVEST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['HARVEST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['FEED'], ['HARVEST'], ['WATER'], ['WEST'], ['WEST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['WATER'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['SOUTH'], 'hands': [['FERTILIZE'], ['NORTH'], ['SOUTH'], ['WEST'], ['WEST'], ['FERTILIZE'], ['DROP'], ['WATER'], ['WEST'], ['NORTH'], ['DROP'], ['SOUTH']], 'market': [['SELL', 'MELON', 12], ['SELL', 'WOOL', 4], ['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['SOUTH'], 'hands': [['WATER'], ['WATER'], ['WEST'], ['FERTILIZE'], ['HARVEST'], ['WATER'], ['PASS'], ['HARVEST'], ['DIG'], ['NORTH'], ['PASS'], ['WEST']], 'market': [['SELL', 'WHEAT', 4], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['SOUTH'], ['WEST'], ['WATER'], ['SOUTH'], ['EAST'], ['PASS'], ['SOUTH'], ['NORTH'], ['WATER'], ['PASS'], ['WEST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['WATER'], 'hands': [['FERTILIZE'], ['SOUTH'], ['WEST'], ['SOUTH'], ['SOUTH'], ['EAST'], ['PASS'], ['SOUTH'], ['WATER'], ['EAST'], ['PASS'], ['WEST']], 'market': [['SELL', 'WHEAT', 4], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 2], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'MELON', 18], ['SELL', 'MILK', 9], ['SELL', 'STRAWBERRY', 12], ['SELL', 'FERTILIZER', 3], ['SELL', 'WHEAT', 11], ['SELL', 'WOOL', 4], ['BUY_PRODUCT', 'WHEAT', 4], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 11], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 11], ['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['SOUTH'], ['NORTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 11], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['HARVEST'], ['WEST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['NORTH'], ['EAST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['CARE'], ['SOUTH'], ['NORTH'], ['NORTH'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['FEED'], ['FEED'], ['FEED'], ['CARE'], ['HARVEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['WATER'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['WEST'], ['CARE'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['NORTH'], ['CARE'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['HARVEST'], 'hands': [['FEED'], ['FEED'], ['COLLECT_FERTILIZER'], ['FEED'], ['CARE'], ['COLLECT_FERTILIZER'], ['WATER'], ['CARE'], ['DIG'], ['NORTH'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['EAST'], ['CARE'], ['WEST'], ['NORTH'], ['NORTH'], ['EAST'], ['SOUTH'], ['EAST'], ['WEST'], ['HARVEST'], ['EAST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['FEED'], 'hands': [['FEED'], ['HARVEST'], ['CARE'], ['WEST'], ['WEST'], ['EAST'], ['WATER'], ['CARE'], ['NORTH'], ['EAST'], ['CARE'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['WEST'], ['WEST'], ['WATER'], ['FEED'], ['EAST'], ['EAST'], ['WEST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['EAST'], ['FEED'], ['WEST'], ['NORTH'], ['SOUTH'], ['WATER'], ['WATER'], ['WEST'], ['NORTH'], ['NORTH'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['HARVEST'], ['EAST'], ['CARE'], ['WEST'], ['WATER'], ['FEED'], ['EAST'], ['NORTH'], ['WATER'], ['HARVEST'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['EAST'], ['HARVEST'], ['WATER'], ['SOUTH'], ['NORTH'], ['WATER'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['HARVEST'], ['EAST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['EAST'], ['COLLECT_FERTILIZER'], ['FERTILIZE'], ['WEST'], ['EAST'], ['NORTH'], ['HARVEST'], ['WEST'], ['SOUTH'], ['EAST'], ['HARVEST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['HARVEST'], ['FEED'], ['WEST'], ['WEST'], ['HARVEST'], ['WATER'], ['NORTH'], ['EAST'], ['WATER'], ['SOUTH'], ['EAST'], ['EAST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['NORTH'], ['FERTILIZE'], ['FERTILIZE'], ['SOUTH'], ['NORTH'], ['WEST'], ['WATER'], ['NORTH'], ['SOUTH'], ['WATER'], ['HARVEST']], 'market': [['SELL', 'WHEAT', 2], ['SELL', 'STRAWBERRY', 2], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['FEED'], 'hands': [['FERTILIZE'], ['NORTH'], ['WATER'], ['WATER'], ['SOUTH'], ['NORTH'], ['WEST'], ['FERTILIZE'], ['NORTH'], ['DROP'], ['EAST'], ['NORTH']], 'market': [['SELL', 'STRAWBERRY', 4], ['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['NORTH'], ['NORTH'], ['NORTH'], ['DIG'], ['EAST'], ['HARVEST'], ['HARVEST'], ['NORTH'], ['PASS'], ['WATER'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['WATER'], 'hands': [['FERTILIZE'], ['WATER'], ['HARVEST'], ['NORTH'], ['WEST'], ['SOUTH'], ['WEST'], ['SOUTH'], ['HARVEST'], ['PASS'], ['HARVEST'], ['WATER']], 'market': [['SELL', 'WHEAT', 4], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['WATER'], ['DIG'], ['SOUTH'], ['HARVEST'], ['SOUTH'], ['WATER'], ['PASS'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['WATER'], ['EAST'], ['SOUTH'], ['WEST'], ['SOUTH'], ['WATER'], ['SOUTH'], ['SOUTH'], ['PASS'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'WHEAT', 4], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['WATER'], ['SOUTH'], ['EAST'], ['SOUTH'], ['WATER'], ['SOUTH'], ['EAST'], ['WEST'], ['WATER'], ['PASS'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'WHEAT', 2], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 2], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'MILK', 15], ['SELL', 'STRAWBERRY', 24], ['SELL', 'FERTILIZER', 7], ['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 2], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['HARVEST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['WEST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['HARVEST'], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['COLLECT_FERTILIZER'], ['WEST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['EAST']], 'market': [['SELL', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['EAST'], ['NORTH'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['FEED'], ['NORTH'], ['FEED'], ['CARE'], ['FEED'], ['COLLECT_FERTILIZER'], ['WEST'], ['WEST'], ['NORTH'], ['EAST'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['NORTH'], ['HARVEST'], ['CARE'], ['NORTH'], ['NORTH'], ['NORTH'], ['WATER'], ['WATER'], ['CARE'], ['HARVEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['FEED'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['CARE'], ['FEED'], ['COLLECT_FERTILIZER'], ['NORTH'], ['SOUTH'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['EAST'], ['SOUTH'], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH'], ['CARE'], ['DIG'], ['WATER'], ['NORTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['CARE'], ['FEED'], ['SOUTH'], ['FEED'], ['WATER'], ['NORTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['FERTILIZE'], ['CARE'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['COLLECT_FERTILIZER'], ['EAST'], ['WEST'], ['CARE'], ['WEST'], ['EAST'], ['FERTILIZE'], ['SOUTH'], ['WATER'], ['NORTH'], ['HARVEST'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['FEED'], ['DROP'], ['HARVEST'], ['WATER'], ['WATER'], ['WEST'], ['SOUTH'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': [['SELL', 'WOOL', 5], ['SELL', 'WOOL', 3]]}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['SOUTH'], ['EAST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['WATER'], ['FERTILIZE'], ['WATER'], ['EAST'], ['NORTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['EAST'], ['EAST'], ['SOUTH'], ['HARVEST'], ['WATER'], ['SOUTH'], ['WATER'], ['WEST'], ['EAST'], ['FERTILIZE'], ['CARE']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['FEED'], ['EAST'], ['SOUTH'], ['WEST'], ['EAST'], ['WEST'], ['WEST'], ['WATER'], ['WATER'], ['SOUTH'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['FERTILIZE'], ['WEST'], ['EAST'], ['HARVEST'], ['WATER'], ['NORTH'], ['EAST'], ['SOUTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['FEED'], ['HARVEST'], ['EAST'], ['WATER'], ['WEST'], ['HARVEST'], ['WEST'], ['NORTH'], ['NORTH'], ['EAST'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['WATER'], ['WATER'], ['SOUTH'], ['HARVEST'], ['SOUTH'], ['HARVEST'], ['WATER'], ['NORTH'], ['HARVEST'], ['EAST'], ['DROP']], 'market': [['SELL', 'MILK', 3], ['SELL', 'WOOL', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['HARVEST'], ['SOUTH'], ['NORTH'], ['FERTILIZE'], ['WATER'], ['EAST'], ['SOUTH'], ['NORTH'], ['SOUTH'], ['WATER'], ['HARVEST'], ['PASS']], 'market': [['SELL', 'WOOL', 3]]}, {'farmer': ['WATER'], 'hands': [['SOUTH'], ['EAST'], ['NORTH'], ['WATER'], ['SOUTH'], ['HARVEST'], ['DIG'], ['WATER'], ['WATER'], ['WEST'], ['SOUTH'], ['PASS']], 'market': []}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['WATER'], ['NORTH'], ['WEST'], ['SOUTH'], ['WATER'], ['SOUTH'], ['EAST'], ['NORTH'], ['WATER'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['WEST'], 'hands': [['SOUTH'], ['SOUTH'], ['FERTILIZE'], ['WATER'], ['WATER'], ['WEST'], ['EAST'], ['EAST'], ['NORTH'], ['SOUTH'], ['WEST'], ['PASS']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FERTILIZE'], ['WEST'], ['WATER'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['EAST'], ['EAST'], ['EAST'], ['SOUTH'], ['WEST'], ['PASS']], 'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 7]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 19], ['SELL', 'MILK', 6], ['SELL', 'WOOL', 8], ['SELL', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['SOUTH'], ['NORTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['HARVEST'], ['SOUTH'], ['EAST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['HARVEST'], ['SOUTH'], ['WEST'], ['EAST'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['FEED'], ['FEED'], ['FEED'], ['CARE'], ['HARVEST'], ['WATER'], ['CARE'], ['WATER'], ['CARE'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['HARVEST'], ['NORTH'], ['NORTH'], ['WEST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['FEED'], ['WEST'], ['FEED'], ['HARVEST'], ['FEED'], ['WEST'], ['CARE'], ['WEST'], ['NORTH'], ['CARE'], ['CARE']], 'market': []}, {'farmer': ['FEED'], 'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['WEST'], ['CARE'], ['WEST'], ['COLLECT_FERTILIZER'], ['WEST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['EAST'], ['WEST'], ['EAST'], ['DIG'], ['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['HARVEST'], ['NORTH'], ['NORTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['FEED'], ['EAST'], ['FEED'], ['CARE'], ['WEST'], ['EAST'], ['NORTH'], ['CARE'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['FEED'], ['CARE'], ['EAST'], ['WEST'], ['EAST'], ['NORTH'], ['EAST'], ['HARVEST'], ['HARVEST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['EAST'], ['SOUTH'], ['HARVEST'], ['FERTILIZE'], ['WATER'], ['SOUTH'], ['HARVEST'], ['FERTILIZE'], ['NORTH'], ['NORTH'], ['NORTH'], ['HARVEST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['FEED'], ['EAST'], ['COLLECT_FERTILIZER'], ['WATER'], ['WEST'], ['EAST'], ['NORTH'], ['WATER'], ['WEST'], ['WATER'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['NORTH'], ['FEED'], ['EAST'], ['NORTH'], ['WATER'], ['EAST'], ['HARVEST'], ['SOUTH'], ['WEST'], ['SOUTH'], ['HARVEST'], ['HARVEST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['NORTH'], ['NORTH'], ['EAST'], ['EAST'], ['SOUTH'], ['FEED'], ['SOUTH'], ['HARVEST'], ['HARVEST'], ['SOUTH'], ['EAST'], ['WEST']], 'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['NORTH'], ['EAST'], ['HARVEST'], ['SOUTH'], ['EAST'], ['WEST'], ['EAST'], ['EAST'], ['SOUTH'], ['WATER'], ['HARVEST']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['SOUTH'], 'hands': [['HARVEST'], ['WEST'], ['DROP'], ['EAST'], ['WATER'], ['WATER'], ['HARVEST'], ['HARVEST'], ['EAST'], ['SOUTH'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'MILK', 6], ['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['NORTH'], ['PASS'], ['WATER'], ['SOUTH'], ['EAST'], ['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['EAST'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['WEST'], 'hands': [['HARVEST'], ['WEST'], ['PASS'], ['NORTH'], ['SOUTH'], ['WATER'], ['EAST'], ['WEST'], ['EAST'], ['SOUTH'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'WHEAT', 3], ['SELL', 'STRAWBERRY', 2], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['DROP'], 'hands': [['NORTH'], ['WATER'], ['PASS'], ['WATER'], ['SOUTH'], ['WEST'], ['EAST'], ['WEST'], ['DROP'], ['NORTH'], ['WATER'], ['SOUTH']], 'market': [['SELL', 'STRAWBERRY', 5], ['SELL', 'WHEAT', 12], ['SELL', 'STRAWBERRY', 2], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PASS'], 'hands': [['WEST'], ['EAST'], ['PASS'], ['SOUTH'], ['HARVEST'], ['WEST'], ['EAST'], ['WEST'], ['PASS'], ['EAST'], ['SOUTH'], ['DROP']], 'market': [['SELL', 'STRAWBERRY', 4], ['SELL', 'WHEAT', 3], ['SELL', 'STRAWBERRY', 2], ['BUY_PRODUCT', 'WHEAT', 12]]}, {'farmer': ['PASS'], 'hands': [['FERTILIZE'], ['WATER'], ['PASS'], ['SOUTH'], ['WATER'], ['WEST'], ['EAST'], ['DROP'], ['PASS'], ['DROP'], ['SOUTH'], ['PASS']], 'market': [['SELL', 'WOOL', 4], ['SELL', 'STRAWBERRY', 5], ['SELL', 'MILK', 3], ['SELL', 'WHEAT', 12], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 10], ['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 15], ['SELL', 'WOOL', 4], ['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 2], ['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 12], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 11], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['NORTH'], ['WEST']], 'market': [['SELL', 'WHEAT', 7], ['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH'], ['NORTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 7]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['EAST'], ['NORTH'], ['NORTH'], ['WEST'], ['FEED'], ['CARE'], ['CARE'], ['WEST'], ['NORTH'], ['NORTH'], ['CARE']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['FEED'], ['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['FEED'], ['NORTH'], ['FEED'], ['CARE'], ['NORTH'], ['WEST'], ['COLLECT_FERTILIZER'], ['CARE'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['FEED'], ['EAST'], ['EAST'], ['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['NORTH'], ['EAST'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['EAST'], ['FEED'], ['EAST'], ['NORTH'], ['NORTH'], ['WATER'], ['NORTH'], ['SOUTH'], ['FERTILIZE'], ['CARE'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['HARVEST'], 'hands': [['COLLECT_FERTILIZER'], ['FEED'], ['SOUTH'], ['FEED'], ['NORTH'], ['NORTH'], ['EAST'], ['WATER'], ['WATER'], ['NORTH'], ['HARVEST'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['WEST'], 'hands': [['EAST'], ['CARE'], ['EAST'], ['COLLECT_FERTILIZER'], ['WATER'], ['WEST'], ['EAST'], ['WEST'], ['EAST'], ['WEST'], ['EAST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['EAST'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH'], ['WEST'], ['WATER'], ['WATER'], ['WEST'], ['WATER'], ['HARVEST'], ['HARVEST'], ['FERTILIZE']], 'market': [['SELL', 'WHEAT', 2]]}, {'farmer': ['CARE'], 'hands': [['EAST'], ['SOUTH'], ['FEED'], ['EAST'], ['FERTILIZE'], ['HARVEST'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['WEST'], ['SOUTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['COLLECT_FERTILIZER'], 'hands': [['EAST'], ['FEED'], ['CARE'], ['EAST'], ['SOUTH'], ['SOUTH'], ['DIG'], ['SOUTH'], ['WATER'], ['WEST'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['EAST'], ['WEST'], ['EAST'], ['WATER'], ['WEST'], ['WEST'], ['EAST'], ['SOUTH'], ['HARVEST'], ['WEST'], ['NORTH'], ['FERTILIZE']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 4]]}, {'farmer': ['FERTILIZE'], 'hands': [['FEED'], ['WEST'], ['HARVEST'], ['NORTH'], ['HARVEST'], ['WEST'], ['DIG'], ['SOUTH'], ['EAST'], ['WATER'], ['NORTH'], ['WATER']], 'market': [['SELL', 'WHEAT', 4], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['WEST'], ['SOUTH'], ['WATER'], ['HARVEST'], ['SOUTH'], ['HARVEST'], ['EAST'], ['SOUTH'], ['WATER'], ['EAST'], ['EAST'], ['WEST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['EAST'], 'hands': [['WEST'], ['SOUTH'], ['SOUTH'], ['EAST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['EAST'], ['WEST']], 'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['DROP'], ['SOUTH'], ['WEST'], ['HARVEST'], ['WEST'], ['SOUTH'], ['WEST'], ['HARVEST'], ['EAST'], ['SOUTH'], ['HARVEST'], ['WEST']], 'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['SOUTH'], ['WEST'], ['WATER'], ['HARVEST'], ['SOUTH'], ['WEST'], ['SOUTH'], ['WATER'], ['WATER'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['FERTILIZE'], ['WEST'], ['WEST'], ['WATER'], ['FERTILIZE'], ['DIG'], ['HARVEST'], ['WEST'], ['WEST'], ['SOUTH'], ['DROP']], 'market': [['SELL', 'MILK', 3], ['SELL', 'WHEAT', 1], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['WATER'], ['DROP'], ['WATER'], ['EAST'], ['EAST'], ['SOUTH'], ['NORTH'], ['WEST'], ['WATER'], ['SOUTH'], ['PASS']], 'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['PASS'], ['NORTH'], ['PASS'], ['SOUTH'], ['EAST'], ['EAST'], ['SOUTH'], ['WATER'], ['HARVEST'], ['SOUTH'], ['WEST'], ['PASS']], 'market': [['SELL', 'WHEAT', 1], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 7], ['BUY_PRODUCT', 'WHEAT', 5]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 24], ['SELL', 'MILK', 6], ['SELL', 'WHEAT', 14], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['HARVEST'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['HARVEST'], ['PICKUP', 'FERTILIZER', 3], ['PICKUP', 'FERTILIZER', 3], ['CARE'], ['HARVEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['EAST']], 'market': [['SELL', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['EAST'], ['WEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['FEED'], ['NORTH'], ['FEED'], ['CARE'], ['FEED'], ['CARE'], ['EAST'], ['HARVEST'], ['HARVEST'], ['NORTH'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['NORTH'], ['HARVEST'], ['WEST'], ['WEST'], ['CARE'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['WEST'], ['SOUTH'], ['WEST'], ['CARE'], ['FEED'], ['CARE'], ['CARE'], ['HARVEST'], ['CARE'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['FEED'], ['SOUTH'], ['FEED'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['EAST'], ['WEST'], ['WEST'], ['NORTH'], ['EAST'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['WEST'], ['HARVEST'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['EAST'], ['SOUTH'], ['CARE'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['SOUTH'], ['SOUTH'], ['HARVEST'], ['EAST'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['EAST'], ['EAST'], ['DROP'], ['COLLECT_FERTILIZER'], ['NORTH'], ['SOUTH'], ['HARVEST'], ['WEST'], ['SOUTH'], ['SOUTH'], ['HARVEST'], ['WATER']], 'market': [['SELL', 'WOOL', 8], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['FEED'], ['FEED'], ['WEST'], ['NORTH'], ['WEST'], ['EAST'], ['WEST'], ['EAST'], ['WEST'], ['SOUTH'], ['EAST'], ['SOUTH']], 'market': []}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['EAST'], ['WEST'], ['NORTH'], ['WATER'], ['CARE'], ['HARVEST'], ['CARE'], ['WEST'], ['SOUTH'], ['EAST'], ['COLLECT_FERTILIZER']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['FEED'], ['WEST'], ['FERTILIZE'], ['WEST'], ['EAST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['WEST'], ['SOUTH'], ['HARVEST'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['SOUTH'], ['HARVEST'], ['WATER'], ['WEST'], ['WATER'], ['WEST'], ['WEST'], ['WATER'], ['WATER'], ['NORTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['WEST'], ['NORTH'], ['WEST'], ['WEST'], ['WEST'], ['NORTH'], ['SOUTH'], ['WEST'], ['NORTH'], ['NORTH'], ['DIG'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['WEST'], ['SOUTH'], ['HARVEST'], ['FERTILIZE'], ['FERTILIZE'], ['NORTH'], ['SOUTH'], ['DROP'], ['NORTH'], ['NORTH'], ['EAST'], ['WATER']], 'market': [['SELL', 'WOOL', 8]]}, {'farmer': ['HARVEST'], 'hands': [['WEST'], ['WEST'], ['SOUTH'], ['WATER'], ['WATER'], ['EAST'], ['EAST'], ['PASS'], ['EAST'], ['NORTH'], ['DIG'], ['EAST']], 'market': []}, {'farmer': ['WATER'], 'hands': [['WEST'], ['WEST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['DIG'], ['EAST'], ['PASS'], ['EAST'], ['EAST'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['FEED'], ['DROP'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['EAST'], ['PASS'], ['EAST'], ['EAST'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'STRAWBERRY', 2]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['PASS'], ['EAST'], ['FERTILIZE'], ['SOUTH'], ['WATER'], ['DROP'], ['PASS'], ['EAST'], ['EAST'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'MILK', 3], ['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['WATER'], ['PASS'], ['HARVEST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['DROP'], ['DROP'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'MILK', 12], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 10]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 18], ['SELL', 'WOOL', 4], ['SELL', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['CARE'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['NORTH'], ['PICKUP', 'WHEAT', 2], ['SOUTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['HARVEST'], ['PICKUP', 'WHEAT', 2], ['CARE'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['SOUTH'], ['EAST'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['EAST'], ['COLLECT_FERTILIZER'], ['FEED'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['CARE'], ['WEST'], ['NORTH'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['PICKUP', 'WHEAT', 2], ['FEED'], ['FEED'], ['NORTH'], ['COLLECT_FERTILIZER'], ['WATER'], ['EAST'], ['NORTH'], ['CARE'], ['NORTH'], ['CARE']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['HARVEST'], ['CARE'], ['HARVEST'], ['SOUTH'], ['NORTH'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['WEST'], ['FEED'], ['NORTH'], ['FEED'], ['WEST'], ['HARVEST'], ['CARE'], ['SOUTH'], ['DIG'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['NORTH'], ['FEED'], ['FEED'], ['SOUTH'], ['WEST'], ['NORTH'], ['WATER'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['SOUTH'], ['NORTH'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['CARE'], ['CARE'], ['CARE'], ['WEST'], ['WATER'], ['WEST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['WATER'], ['WEST'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['HARVEST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['FERTILIZE'], ['WEST'], ['CARE'], ['EAST'], ['WEST'], ['WEST'], ['COLLECT_FERTILIZER']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['EAST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['WATER'], ['WEST'], ['WATER'], ['COLLECT_FERTILIZER'], ['WEST'], ['WATER'], ['HARVEST'], ['SOUTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['EAST'], ['SOUTH'], ['EAST'], ['SOUTH'], ['WATER'], ['HARVEST'], ['NORTH'], ['WEST'], ['HARVEST'], ['WEST'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['EAST'], ['WEST'], ['FEED'], ['WEST'], ['SOUTH'], ['WEST'], ['HARVEST'], ['SOUTH'], ['WEST'], ['WEST'], ['EAST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['WEST'], ['FEED'], ['WATER'], ['EAST'], ['WEST'], ['WEST'], ['DIG'], ['SOUTH'], ['SOUTH'], ['WATER'], ['WEST'], ['HARVEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WEST'], ['EAST'], ['SOUTH'], ['FEED'], ['HARVEST'], ['WEST'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['HARVEST'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['DROP'], ['FEED'], ['FERTILIZE'], ['EAST'], ['SOUTH'], ['HARVEST'], ['DIG'], ['NORTH'], ['SOUTH'], ['WATER'], ['SOUTH'], ['WATER']], 'market': [['SELL', 'WOOL', 4], ['SELL', 'MILK', 3], ['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['FEED'], 'hands': [['PASS'], ['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['NORTH'], ['EAST'], ['SOUTH'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['NORTH'], 'hands': [['WEST'], ['DIG'], ['WATER'], ['WATER'], ['CARE'], ['SOUTH'], ['NORTH'], ['DIG'], ['WATER'], ['WEST'], ['WATER'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WATER'], 'hands': [['PASS'], ['SOUTH'], ['FERTILIZE'], ['WEST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['WATER'], ['SOUTH'], ['NORTH'], ['WATER'], ['SOUTH'], ['SOUTH']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['SOUTH'], ['NORTH'], ['WEST'], ['WEST'], ['EAST'], ['NORTH'], ['SOUTH'], ['EAST'], ['EAST'], ['SOUTH'], ['WEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['PASS'], ['SOUTH'], ['NORTH'], ['WEST'], ['EAST'], ['EAST'], ['NORTH'], ['SOUTH'], ['EAST'], ['EAST'], ['WATER'], ['WEST']], 'market': []}, {'farmer': ['WEST'], 'hands': [['PASS'], ['WEST'], ['WATER'], ['WEST'], ['EAST'], ['EAST'], ['EAST'], ['WEST'], ['DROP'], ['EAST'], ['SOUTH'], ['WEST']], 'market': [['SELL', 'WOOL', 4], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 10]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 20], ['SELL', 'MILK', 6], ['SELL', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['FEED'], 'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 15], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['PICKUP', 'WHEAT', 2], ['WEST'], ['NORTH'], ['NORTH'], ['NORTH']], 'market': [['SELL', 'WHEAT', 13], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['PICKUP', 'WHEAT', 2], ['EAST'], ['HARVEST'], ['WEST'], ['WEST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE']]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['SOUTH'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['PICKUP', 'WHEAT', 2], ['WEST'], ['FEED'], ['FEED'], ['EAST'], ['HARVEST'], ['WEST'], ['WEST'], ['NORTH'], ['EAST'], ['WEST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['SOUTH'], ['HARVEST'], ['WEST'], ['COLLECT_FERTILIZER'], ['EAST'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['NORTH'], ['WEST'], ['NORTH'], ['EAST'], ['DROP'], ['SOUTH'], ['HARVEST'], ['NORTH'], ['HARVEST'], ['WEST']], 'market': [['SELL', 'MILK', 6]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['FEED'], ['COLLECT_FERTILIZER'], ['FEED'], ['EAST'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['WEST'], ['NORTH'], ['HARVEST']], 'market': []}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['WEST'], ['COLLECT_FERTILIZER'], ['EAST'], ['COLLECT_FERTILIZER'], ['HARVEST'], ['NORTH'], ['SOUTH'], ['NORTH'], ['HARVEST'], ['DIG'], ['NORTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 3]]}, {'farmer': ['PICKUP', 'WHEAT', 2], 'hands': [['FEED'], ['EAST'], ['EAST'], ['NORTH'], ['NORTH'], ['EAST'], ['SOUTH'], ['WEST'], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['EAST'], 'hands': [['HARVEST'], ['EAST'], ['EAST'], ['WEST'], ['WEST'], ['EAST'], ['WATER'], ['WEST'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['COLLECT_FERTILIZER'], ['FEED'], ['EAST'], ['WEST'], ['NORTH'], ['COLLECT_FERTILIZER'], ['NORTH'], ['HARVEST'], ['NORTH'], ['WEST'], ['WATER']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['SOUTH'], ['NORTH'], ['FEED'], ['HARVEST'], ['WEST'], ['SOUTH'], ['WEST'], ['NORTH'], ['DIG'], ['COLLECT_FERTILIZER'], ['WEST']], 'market': [['BUY_PRODUCT', 'WHEAT', 2]]}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['FEED'], ['WEST'], ['NORTH'], ['DIG'], ['SOUTH'], ['HARVEST'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['WATER']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['SOUTH'], ['EAST'], ['WEST'], ['WEST'], ['SOUTH'], ['WEST'], ['SOUTH'], ['NORTH'], ['WATER'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 2], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['EAST'], 'hands': [['FERTILIZE'], ['FEED'], ['SOUTH'], ['WEST'], ['SOUTH'], ['WEST'], ['EAST'], ['DIG'], ['WEST'], ['DROP'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['WATER'], ['SOUTH'], ['SOUTH'], ['DIG'], ['EAST'], ['SOUTH'], ['EAST'], ['SOUTH'], ['WATER'], ['CARE'], ['EAST']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['SOUTH'], 'hands': [['SOUTH'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['WATER'], ['SOUTH'], ['EAST'], ['WATER'], ['SOUTH'], ['PASS'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['FEED'], 'hands': [['FERTILIZE'], ['WEST'], ['WEST'], ['SOUTH'], ['EAST'], ['SOUTH'], ['EAST'], ['EAST'], ['SOUTH'], ['PASS'], ['SOUTH']], 'market': [['SELL', 'FERTILIZER', 2]]}, {'farmer': ['WEST'], 'hands': [['NORTH'], ['DROP'], ['FERTILIZE'], ['SOUTH'], ['WATER'], ['SOUTH'], ['HARVEST'], ['WATER'], ['SOUTH'], ['PASS'], ['WATER']], 'market': [['SELL', 'MILK', 3], ['BUY_PRODUCT', 'WHEAT', 1]]}, {'farmer': ['WEST'], 'hands': [['WEST'], ['PASS'], ['WATER'], ['SOUTH'], ['WEST'], ['WEST'], ['WATER'], ['EAST'], ['EAST'], ['PASS'], ['WEST']], 'market': []}, {'farmer': ['PASS'], 'hands': [['WATER'], ['PASS'], ['WEST'], ['EAST'], ['WEST'], ['FERTILIZE'], ['WEST'], ['SOUTH'], ['EAST'], ['PASS'], ['WATER']], 'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 2]]}, {'farmer': ['PASS'], 'hands': [['NORTH'], ['PASS'], ['WATER'], ['EAST'], ['WEST'], ['WATER'], ['NORTH'], ['WATER'], ['DROP'], ['PASS'], ['NORTH']], 'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 10]]}, {'farmer': ['EAST'], 'hands': [], 'market': [['SELL', 'STRAWBERRY', 16], ['SELL', 'MILK', 4], ['SELL', 'FERTILIZER', 5], ['SELL', 'WHEAT', 34], ['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['HARVEST'], 'hands': [['NORTH'], ['SOUTH'], ['NORTH'], ['WEST']], 'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]}, {'farmer': ['NORTH'], 'hands': [['NORTH'], ['SOUTH'], ['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['NORTH'], ['NORTH']], 'market': [['HIRE'], ['HIRE']]}, {'farmer': ['HARVEST'], 'hands': [['EAST'], ['WEST'], ['EAST'], ['HARVEST'], ['EAST'], ['SOUTH'], ['NORTH'], ['EAST'], ['NORTH'], ['SOUTH']], 'market': []}, {'farmer': ['NORTH'], 'hands': [['HARVEST'], ['HARVEST'], ['HARVEST'], ['NORTH'], ['EAST'], ['WEST'], ['EAST'], ['SOUTH'], ['NORTH'], ['SOUTH']], 'market': []}, {'farmer': ['EAST'], 'hands': [['SOUTH'], ['WEST'], ['EAST'], ['NORTH'], ['HARVEST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['HARVEST'], 'hands': [['SOUTH'], ['HARVEST'], ['HARVEST'], ['WEST'], ['WEST'], ['HARVEST'], ['WEST'], ['SOUTH'], ['WEST'], ['WEST']], 'market': []}, {'farmer': ['SOUTH'], 'hands': [['DROP'], ['WEST'], ['SOUTH'], ['HARVEST'], ['WEST'], ['WEST'], ['CARE'], ['PASS'], ['WEST'], ['SOUTH']], 'market': [['SELL', 'WOOL', 4]]}, {'farmer': ['SOUTH'], 'hands': [['COLLECT_FERTILIZER'], ['HARVEST'], ['WEST'], ['WEST'], ['DROP'], ['WEST'], ['PASS'], ['PASS'], ['SOUTH'], ['SOUTH']], 'market': [['SELL', 'MILK', 3]]}, {'farmer': ['WEST'], 'hands': [['PASS'], ['NORTH'], ['WEST'], ['HARVEST'], ['PASS'], ['HARVEST'], ['PASS'], ['PASS'], ['SOUTH'], ['HARVEST']], 'market': []}, {'farmer': ['DROP'], 'hands': [['PASS'], ['NORTH'], ['DROP'], ['SOUTH'], ['PASS'], ['NORTH'], ['PASS'], ['PASS'], ['EAST'], ['NORTH']], 'market': [['SELL', 'WOOL', 16], ['SELL', 'MILK', 3]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['EAST'], ['PASS'], ['SOUTH'], ['PASS'], ['NORTH'], ['PASS'], ['PASS'], ['EAST'], ['NORTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['EAST'], ['PASS'], ['EAST'], ['PASS'], ['NORTH'], ['PASS'], ['PASS'], ['EAST'], ['NORTH']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['EAST'], ['PASS'], ['EAST'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['CARE'], ['NORTH']], 'market': [['SELL', 'STRAWBERRY', 2]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['DROP'], ['PASS'], ['EAST'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['COLLECT_FERTILIZER'], ['EAST']], 'market': [['SELL', 'STRAWBERRY', 5]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['EAST']], 'market': [['SELL', 'STRAWBERRY', 2]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['DROP'], ['PASS'], ['DROP'], ['PASS'], ['PASS'], ['PASS'], ['DROP']], 'market': [['SELL', 'STRAWBERRY', 9], ['SELL', 'MILK', 3]]}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}, {'farmer': ['PASS'], 'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']], 'market': []}]

# === DOMAIN CONSTANTS ===
BASE_PRICE = {
    "WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120,
    "MELON": 250, "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100,
}

SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}

SENSITIVITY = {
    "WHEAT": 0.8, "CARROT": 1.2, "TOMATO": 1.0, "STRAWBERRY": 1.5,
    "MELON": 2.0, "EGG": 0.9, "MILK": 1.8, "WOOL": 2.0, "FERTILIZER": 1.0,
}

TERMINAL_PRODUCTS = (
    "MELON", "STRAWBERRY", "MILK", "WOOL", "TOMATO",
    "EGG", "CARROT", "WHEAT", "FERTILIZER",
)

PREMIUM_ITEMS = frozenset({"STRAWBERRY", "MELON", "MILK", "WOOL"})
TERMINAL_START = 696
MAX_ORDERS = 10


# === HELPERS ===
def _get(d, k, default):
    if isinstance(d, dict):
        return d.get(k, default)
    return getattr(d, k, default)


def _prices(obs):
    market = _get(obs, "market", {}) or {}
    return _get(market, "prices", {}) or {}


def _shed(obs):
    private = _get(obs, "private", {}) or {}
    return _get(private, "shed", {}) or {}


def _iter_tiles(farm):
    for row in (_get(farm, "tiles", []) or []):
        for tile in row:
            yield tile


def _move_toward(pos, target, tiles):
    x, y = pos[0], pos[1]
    tx, ty = target[0], target[1]
    size = len(tiles)
    choices = []
    if tx < x:
        choices.append(("WEST", (x - 1, y)))
    if tx > x:
        choices.append(("EAST", (x + 1, y)))
    if ty < y:
        choices.append(("NORTH", (x, y - 1)))
    if ty > y:
        choices.append(("SOUTH", (x, y + 1)))
    for op, (nx, ny) in choices:
        if 0 <= nx < size and 0 <= ny < size and tiles[ny][nx] != "LOCKED":
            return [op]
    return ["PASS"]


def _shed_tiles(size):
    half = size // 2
    return [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]


# === OVERLAY 1: HAND COUNT ADAPTATION ===
def _adapt_hands(action, obs):
    """Pad or trim the hands action list to match the actual hand count."""
    farms = _get(obs, "farms", []) or []
    player = int(_get(obs, "player", 0) or 0)
    if player >= len(farms):
        return
    n = len(_get(farms[player], "hands", []) or [])
    hands = list(action.get("hands", []) or [])[:n]
    hands += [["PASS"] for _ in range(n - len(hands))]
    action["hands"] = hands


# === OVERLAY 2: OPPONENT EXPOSURE MODELING ===
def _opponent_exposure(obs):
    """Estimate the opponent's production capacity per item."""
    farms = _get(obs, "farms", []) or []
    player = int(_get(obs, "player", 0) or 0)
    exposure = {item: 0 for item in BASE_PRICE}
    if len(farms) < 2:
        return exposure
    opponent = farms[1 - player]
    for tile in _iter_tiles(opponent):
        if not isinstance(tile, dict):
            continue
        animal = _get(tile, "animal", None)
        crop = _get(tile, "crop", None)
        if animal == "COW":
            exposure["MILK"] += 6
        elif animal == "SHEEP":
            exposure["WOOL"] += 6
        elif animal == "GOOSE":
            exposure["EGG"] += 4
        elif crop == "MELON":
            exposure["MELON"] += 6
        elif crop == "STRAWBERRY":
            exposure["STRAWBERRY"] += 4
        elif crop == "TOMATO":
            exposure["TOMATO"] += 4
        elif crop == "CARROT":
            exposure["CARROT"] += 4
        elif crop == "WHEAT":
            exposure["WHEAT"] += 6
    return exposure


# === OVERLAY 3: SELL PRIORITY AND REORDERING ===
def _sell_priority(order, obs, exposure):
    """Score a sell order by value plus opponent pressure."""
    prices = _prices(obs)
    item = order[1]
    quantity = max(0, int(order[2] or 0))
    price = int(prices.get(item, BASE_PRICE.get(item, 1)) or 0)
    pressure = float(exposure.get(item, 0) or 0)
    return (
        price * quantity
        + pressure * SENSITIVITY.get(item, 1.0) * max(1, price) * 0.18
    )


def _reorder_sells(action, obs, exposure):
    """Re sort SELL orders by descending priority. Non SELL orders stay in place."""
    orders = list(action.get("market", []) or [])
    positions = [
        i for i, o in enumerate(orders)
        if isinstance(o, list) and len(o) >= 3 and o[0] == "SELL" and int(o[2] or 0) > 0
    ]
    if len(positions) >= 2:
        sells = [orders[i] for i in positions]
        sells.sort(key=lambda o: (-_sell_priority(o, obs, exposure), o[1]))
        for i, o in zip(positions, sells):
            orders[i] = o
    action["market"] = orders[:MAX_ORDERS]


# === OVERLAY 4: TERMINAL LIQUIDATION ===
def _existing_sells(orders):
    """Count existing sell quantities per item in the current orders."""
    existing = {}
    for order in orders:
        if isinstance(order, list) and len(order) >= 3 and order[0] == "SELL":
            existing[order[1]] = existing.get(order[1], 0) + int(order[2] or 0)
    return existing


def _terminal_liquidate(action, obs, step):
    """Fill remaining market slots with shed items by descending price times quantity."""
    if step < TERMINAL_START:
        return
    orders = list(action.get("market", []) or [])
    if len(orders) >= MAX_ORDERS:
        return
    shed = _shed(obs)
    prices = _prices(obs)
    existing = _existing_sells(orders)
    choices = []
    for item in TERMINAL_PRODUCTS:
        quantity = max(0, int(shed.get(item, 0) or 0) - existing.get(item, 0))
        if quantity <= 0:
            continue
        price = int(prices.get(item, BASE_PRICE[item]) or 0)
        # Batch size control for premium goods to avoid crashing price to floor
        if item in PREMIUM_ITEMS:
            quantity = min(quantity, 5)
        else:
            quantity = min(quantity, 20)
        choices.append((price * quantity, item, quantity))
    choices.sort(reverse=True)
    for _, item, quantity in choices:
        if len(orders) >= MAX_ORDERS:
            break
        orders.append(["SELL", item, quantity])
    action["market"] = orders[:MAX_ORDERS]


# === OVERLAY 5: TERMINAL HARVESTING ===
def _terminal_harvest(action, obs, step):
    """Redirect PASS actions to productive work in the terminal phase.
    Replaces PASS with HARVEST, DROP, or MOVE toward harvestable tiles."""
    if step < TERMINAL_START:
        return
    farms = _get(obs, "farms", []) or []
    player = int(_get(obs, "player", 0) or 0)
    if player >= len(farms):
        return
    farm = farms[player]
    private = _get(obs, "private", {}) or {}
    tiles = _get(farm, "tiles", []) or []
    size = len(tiles)
    if size == 0:
        return
    sheds = set(_shed_tiles(size))

    # Find all harvestable tiles
    harvestable = set()
    for y, row in enumerate(tiles):
        for x, tile in enumerate(row):
            if isinstance(tile, dict) and int(_get(tile, "yield_units", 0) or 0) > 0:
                harvestable.add((x, y))

    # Collect all unit positions and inventories
    positions = [tuple(_get(farm, "farmer", [0, 0]))]
    positions += [tuple(h) for h in (_get(farm, "hands", []) or [])]
    inventories = list(_get(private, "inventories", []) or [])
    while len(inventories) < len(positions):
        inventories.append({})

    # Process farmer and hands
    farmer_action = action.get("farmer", ["PASS"])
    hands_actions = list(action.get("hands", []) or [])

    all_actions = [farmer_action] + hands_actions
    claimed = set()

    for i, (pos, inv, act) in enumerate(zip(positions, inventories, all_actions)):
        if not isinstance(act, list) or not act:
            continue
        if act[0] != "PASS":
            claimed.add(pos)
            continue

        x, y = pos
        tile = tiles[y][x] if 0 <= y < size and 0 <= x < size else None
        load = sum(max(0, int(v or 0)) for v in (inv or {}).values())

        # Priority 1: DROP if carrying items and at shed
        if load > 0 and pos in sheds:
            all_actions[i] = ["DROP"]
            claimed.add(pos)
            continue

        # Priority 2: HARVEST if standing on harvestable tile
        if pos in harvestable and pos not in claimed:
            all_actions[i] = ["HARVEST"]
            harvestable.discard(pos)
            claimed.add(pos)
            continue

        # Priority 3: MOVE toward nearest harvestable tile
        if harvestable and load == 0:
            target = min(harvestable, key=lambda q: (abs(q[0] - x) + abs(q[1] - y), q[1], q[0]))
            if target != pos:
                move = _move_toward(pos, target, tiles)
                if move != ["PASS"]:
                    all_actions[i] = move
                    claimed.add(pos)
                    continue

        # Priority 4: MOVE toward shed if carrying items
        if load > 0:
            target = min(sheds, key=lambda q: abs(q[0] - x) + abs(q[1] - y))
            if target != pos:
                move = _move_toward(pos, target, tiles)
                if move != ["PASS"]:
                    all_actions[i] = move
                    claimed.add(pos)
                    continue

        # Priority 5: COLLECT_FERTILIZER if available
        if isinstance(tile, dict) and _get(tile, "fertilizer_available", False):
            all_actions[i] = ["COLLECT_FERTILIZER"]
            claimed.add(pos)
            continue

    action["farmer"] = all_actions[0]
    action["hands"] = all_actions[1:]


# === OVERLAY 6: OPPONENT PROFILE DETECTION ===
_MODE = "unknown"
_LAST_STEP = -1

def _farm_signature(farm):
    sig = {"PLANT": 0, "COOP": 0, "PASTURE": 0, "ANIMAL": 0, "WEED": 0}
    for tile in _iter_tiles(farm):
        if not isinstance(tile, dict):
            continue
        kind = _get(tile, "kind", None)
        if kind in sig:
            sig[kind] += 1
        if _get(tile, "animal", None):
            sig["ANIMAL"] += 1
    return sig


def _detect_profile(obs, step):
    farms = _get(obs, "farms", []) or []
    player = int(_get(obs, "player", 0) or 0)
    if len(farms) < 2:
        return "unknown"
    opponent = farms[1 - player]
    hands = len(_get(opponent, "hands", []) or [])
    quadrants = tuple(_get(opponent, "unlocked_quadrants", []) or ())
    signature = _farm_signature(opponent)
    money = float(_get(opponent, "money", 0) or 0)
    if (
        step >= 2
        and hands == 3
        and len(quadrants) == 1
        and signature.get("PASTURE", 0) == 1
        and signature.get("COOP", 0) == 0
        and signature.get("PLANT", 0) == 0
        and signature.get("ANIMAL", 0) == 0
        and 350 <= money <= 850
    ):
        return "kaito_clone"
    if (
        step >= 1
        and hands == 2
        and len(quadrants) == 1
        and signature.get("PLANT", 0) == 0
        and signature.get("ANIMAL", 0) == 0
        and 900 <= money <= 1550
    ):
        return "tape_like"
    if hands >= 6 and len(quadrants) >= 2:
        return "scheduler"
    return "other"


def _get_mode(obs, step):
    global _MODE, _LAST_STEP
    if step == 0 or step <= _LAST_STEP:
        _MODE = "unknown"
    _LAST_STEP = step
    if step <= 5:
        detected = _detect_profile(obs, step)
        if detected in ("kaito_clone", "tape_like"):
            _MODE = detected
        elif _MODE == "unknown":
            _MODE = detected
    return _MODE


# === OVERLAY 7: FRONT RUN INJECTION ===
EXPOSURE_THRESHOLD = {
    "WHEAT": 30, "CARROT": 20, "TOMATO": 16, "STRAWBERRY": 16,
    "MELON": 18, "EGG": 20, "MILK": 12, "WOOL": 12, "FERTILIZER": 999,
}

# Items that are NEVER front-run because they are needed for the feed/care cycle
NEVER_FRONT_RUN = frozenset({"WHEAT", "FERTILIZER"})


def _scheduled_sales(step, horizon):
    """Scan future trace actions for planned sells within the horizon."""
    planned = {}
    for offset in range(1, min(horizon + 1, 720 - step)):
        future_step = step + offset
        if future_step >= len(TRACE_ACTIONS):
            break
        for order in TRACE_ACTIONS[future_step].get("market", []) or []:
            if not (isinstance(order, list) and len(order) >= 3 and order[0] == "SELL"):
                continue
            item = order[1]
            qty = int(order[2] or 0)
            if item not in planned:
                planned[item] = (offset, qty)
            else:
                _, q = planned[item]
                planned[item] = (offset, q + qty)
    return planned


def _append_front_run(action, obs, step, exposure, mode):
    """Add an extra sell for items that will be sold soon.
    When the opponent is a tape agent, front run aggressively.
    When the opponent is unknown, front run only when prices are favorable.
    Never front run WHEAT or FERTILIZER (needed for feed and animal care)."""
    orders = list(action.get("market", []) or [])
    if len(orders) >= MAX_ORDERS:
        return
    shed = _shed(obs)
    prices = _prices(obs)
    existing = _existing_sells(orders)
    exact = mode in ("kaito_clone", "tape_like")
    horizon = 1 if exact else 2
    planned = _scheduled_sales(step, horizon)
    support = _shop_support(obs)
    choices = []
    for item, (distance, planned_qty) in planned.items():
        # Never front run wheat or fertilizer
        if item in NEVER_FRONT_RUN:
            continue
        available = max(0, int(shed.get(item, 0) or 0) - existing.get(item, 0))
        if available <= 0:
            continue
        quantity = min(available, planned_qty)
        if quantity <= 0:
            continue
        price = int(prices.get(item, BASE_PRICE[item]) or 0)
        base = BASE_PRICE[item]
        pressure = float(exposure.get(item, 0) or 0)
        crowded = pressure >= EXPOSURE_THRESHOLD.get(item, 10**9)
        scarce = price >= int(round(base * 1.12))
        demand_relief = support.get(item, 0)
        if not exact:
            # Conservative: only front run if crowded or scarce
            if not (crowded or scarce):
                continue
            # And price must be above shop adjusted reservation
            reservation = int(round(base * (0.62 + 0.04 * demand_relief)))
            if price < reservation:
                continue
            # Limit batch size
            if item in PREMIUM_ITEMS:
                quantity = min(quantity, 6)
            else:
                quantity = min(quantity, 10)
        else:
            # Aggressive: front run regardless of price, but limit batch size
            if item in PREMIUM_ITEMS:
                quantity = min(quantity, 6)
            else:
                quantity = min(quantity, 10)
        urgency = (
            price * quantity
            + pressure * SENSITIVITY.get(item, 1.0) * price * 0.22
            + max(0, horizon + 1 - distance) * base
        )
        choices.append((urgency, item, quantity))
    if not choices:
        return
    choices.sort(reverse=True)
    # Add at most 1 front run sell
    if len(orders) < MAX_ORDERS:
        _, item, quantity = choices[0]
        orders.append(["SELL", item, quantity])
    action["market"] = orders[:MAX_ORDERS]


def _shop_support(obs):
    """Count how many unlocked shops consume each item."""
    town = _get(obs, "town", {}) or {}
    shops = _get(town, "unlocked_shops", []) or []
    support = {item: 0 for item in BASE_PRICE}
    for shop in shops:
        for item in SHOP_PRODUCTS.get(shop, ()):
            support[item] = support.get(item, 0) + 1
    return support


# === TOP LEVEL AGENT ===
def agent(obs, config=None):
    step = int(_get(obs, "step", 0) or 0)

    # Replay the trace
    action = copy.deepcopy(TRACE_ACTIONS[min(step, 719)])

    # Overlay 1: adapt hand count
    _adapt_hands(action, obs)

    # Overlay 2: opponent exposure (for terminal liquidate priority)
    exposure = _opponent_exposure(obs)

    # Overlay 3: terminal liquidation (fill remaining market slots)
    _terminal_liquidate(action, obs, step)

    # Overlay 4: terminal harvesting (redirect PASS to harvest/drop/move)
    _terminal_harvest(action, obs, step)

    return action
